# NK13 Analysis VII - Multimer figures
- Plot 1: Quantification of multimer+ cells in CSF and PB using virus_extended'
- Plot 2: UMAPs
- Plot 3: Stacked bargraphs with L3_ann_final (proportion of cells)
- Plot 4: % in CSF of EBV-specific clones
- Plot 5: Stacked bar graphs - number of deorphanized clones
- Plot 6: Stacked bar graphs - PB/CSF
- Plot 7: Stacked bar graphs - MS vs NIC
- Plot 8: B-LCL reactive clones in the multimer sort data
- Plot 9: LCL-reactive clones among CSF-biased and PB-biased T cells
- Plot 10: LCL react + mult + compu ann plots
- Plot 11: Upset plot
- Plot 12: Quantifications of antigen-specific cells in PB and CSF
- Plot 13: Number of clonotypes common between datasets (mult vs main)

In [ ]:
import anndata as ad
import scanpy as sc
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import muon as mu
import scvi
import torch

# from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon
import scipy.stats as stats
# import string
from matplotlib.patches import Patch


# Helps to define plotting area

import matplotlib.pyplot as plt
import seaborn.objects as so
from scipy.stats import kruskal
import scikit_posthocs as sp
from skbio.diversity.alpha import gini_index, shannon, simpson
# from matplotlib import rcParams 
# from matplotlib.pyplot import rc_context
# import matplotlib.gridspec as gridspec
# import matplotlib.colors as mcolors
# from matplotlib import colors
# # from matplotlib_venn import venn2
# from scipy.stats import spearmanr
# import matplotlib.patches as mpatches
# from pathlib import Path

#Pseudobulk gene expression
import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#Set seaborn theme
sb.set_style("ticks")
# custom_params = {"axes.spines.right": False, "axes.spines.top": False}
# sb.set_theme(style="ticks", rc=custom_params)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [ ]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("fontTools").setLevel(logging.WARNING)

# Loading CD8 T cell data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_CD8_annotated.h5mu")
len(mdata)

# Short TCR.10 investigation
Predicted by ImmuneWatch to recognize EBV - we validated that

In [ ]:
## Short TCR.10 investigation (Pt8-3399), which was predicted by ImmuneWatch (and we validated it in the internship)
IMW_ann = pd.read_table(sep="\t", filepath_or_buffer="../../03_annotation/12_computational_agspecificity_annotation/Immunewatch_1.4.7/2027_7_full_CD8T_mults_main_predictions.tsv")
print(IMW_ann.columns) 
#Cutoff suggestion for the Score is 0.2
IMW_ann = IMW_ann[IMW_ann["Score"] >= 0.2]
IMW_ann[IMW_ann["clone_id"] == "Pt8-3399"]

# Load full IMW/VDJdb ect annotations

In [ ]:
total_ann = pd.read_csv("../../06_csv_outputs/Clones/2026_7_total_VirusAnnotationsPerClone.csv")
total_ann.head(5)

total_ann[["Virus_final", "Antigen_final"]].value_counts()

In [ ]:
mult_ann = pd.read_csv("../../03_annotation/11_multimer_experiment_ann/2_MultPostClones.csv")
# mult_ann[mult_ann["Virus"] != mult_ann["Virus.1"]]
mult_ann.head(5)

mult_ann[["Virus", "Antigen"]].value_counts()

## Annotate .obs

In [ ]:
#Annotate "Virus_extended" to contain the full Virus annotations combining multimer and databases searches. 
clone_dict = total_ann.set_index("clone_handle")["Virus_final"].to_dict()
mdata["gex"].obs["Virus_extended"] = mdata["gex"].obs["clone_handle"].map(clone_dict)

#Annotate "Antigen_extended" to contain the full Antigen annotations like for Virus_extended
clone_dict = total_ann.set_index("clone_handle")["Antigen_final"].to_dict()
mdata["gex"].obs["Antigen_extended"] = mdata["gex"].obs["clone_handle"].map(clone_dict)

#Virus, i.e. multimer virus make a dictionary, then map it's values to .obs
clone_dict = mult_ann.set_index("clone_handle")["Virus"].to_dict()
mdata["gex"].obs["Virus"] = mdata["gex"].obs["clone_handle"].map(clone_dict)

#Add antigen too
clone_dict = mult_ann.set_index("clone_handle")["Antigen"].to_dict()
mdata["gex"].obs["Antigen"] = mdata["gex"].obs["clone_handle"].map(clone_dict)

## Should CMV-specific cells in CMV(-) donors be excluded? 
mdata["gex"].obs["Virus_extended"] = np.where((mdata["gex"].obs["donor"].isin(["Pt4", "Pt5", "Pt8", "Pt11", "Pt17", "Pt20", "Pt21"])) & (mdata["gex"].obs["Virus_extended"] == "CMV"), np.nan,
                                              mdata["gex"].obs["Virus_extended"])

In [ ]:
mdata["gex"].obs["Virus_extended"].value_counts()

In [ ]:
mdata["gex"].obs["Antigen_extended"].value_counts()

# Plot 1: Quantification of multimer+ cells in CSF and PB using virus_extended

In [ ]:
## check
obs_df = mdata["gex"].obs.copy()
obs_df[(obs_df["Virus_extended"] == "EBV") & (obs_df["Compartment"] == "PB") & (obs_df["donor"]=="Pt8")]

## check
obs_df = mdata["gex"].obs.copy()
obs_df[(obs_df["Virus_extended"] == "EBV") & (obs_df["Compartment"] == "CSF") & (obs_df["donor"]=="Pt8")]

## check
obs_df = mdata["gex"].obs.copy()
# obs_df[(obs_df["clone_handle"] == "Pt8-3399")]

In [ ]:
## Filter
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["total_clone_count"] > 1]
obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "FLU-A", "EBV"])]

#Drop unused categories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] =obs_df[col].cat.remove_unused_categories()

## Count and pivot to get counts per clone in a wide format
count_df = obs_df.groupby(["clone_handle", "Compartment"]).size().reset_index(name="count")
count_df = count_df.pivot(index="clone_handle", columns="Compartment", values="count")
count_df = count_df.reset_index()

## Annotate
d_ann = obs_df[["clone_handle", "donor"]].drop_duplicates()
V_ann = obs_df[["clone_handle", "Virus_extended"]].drop_duplicates()
count_df = count_df.merge(d_ann, how="left").merge(V_ann, how="left")
count_df["clone_total"] = count_df["CSF"] + count_df["PB"]
count_df["clone_csf_pct"] = 100*count_df["CSF"] / count_df["clone_total"]
count_df["mean_clone_csf_pct"] = count_df.groupby(["donor", "Virus_extended"])["clone_csf_pct"].transform("mean")

## Report
# clone_report = count_df[count_df["clone_handle"] == "Pt1-10"]
# print(clone_report)

## Aggregate per donor
count_df["sum_csf"] = count_df.groupby(["donor", "Virus_extended"])["CSF"].transform("sum")
count_df["sum_pb"] = count_df.groupby(["donor", "Virus_extended"])["PB"].transform("sum")
count_df["sum_total"] = count_df["sum_csf"] + count_df["sum_pb"]
count_df["percentage_csf"] = 100*count_df["sum_csf"] / count_df["sum_total"]

## clean up
sum_df = count_df[["donor", "Virus_extended", "sum_csf", "sum_pb", "sum_total", "percentage_csf", "mean_clone_csf_pct"]].drop_duplicates()

## annotate
c_ann = obs_df[[ "donor", "catpat"]].drop_duplicates()
sum_df = sum_df.merge(c_ann, how="left")

## make categories / order
sum_df["catpat"] = pd.Categorical(sum_df["catpat"], ordered=True, categories=["NIC", "MS"])
sum_df["Virus_extended"] = pd.Categorical(sum_df["Virus_extended"], ordered=True, categories=["CMV", "FLU-A", "EBV"])

# colors
catpat_palette = {
    "NIC": "#FFFFFF", 
    "MS":  "#000000" 
}

white_palette = {
    "MS": "white",
    "NIC": "white",
}

fig, ax = plt.subplots(figsize=(4, 4))

sb.barplot(
    data=sum_df,
    x="Virus_extended",
    y="mean_clone_csf_pct",
    # y="percentage_csf",
    hue="catpat",
    palette=white_palette,
    edgecolor="black",
    estimator="mean",
    errorbar="sd",
    linewidth=0.5,
    capsize=0.15,
    ax=ax,
)

sb.stripplot(
    data=sum_df,
    x="Virus_extended",
    y="mean_clone_csf_pct",
    # y="percentage_csf",
    hue="catpat",
    linewidth=0.5,
    jitter=0.25,
    edgecolor="black",
    dodge=True,
    palette=catpat_palette,
    size=7,
    ax=ax,
)

# Remove duplicate legends
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:], labels[2:], title="")

ax.set_ylim(-1, 110)
ax.set_ylabel("Mean clone-wise freq\nof cells in CSF")
ax.set_xlabel("")
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/44_VirusSpecific_in_CSF.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

sb.barplot(
    data=sum_df,
    x="Virus_extended",
    y="percentage_csf",
    # y="percentage_csf",
    hue="catpat",
    palette=white_palette,
    edgecolor="black",
    estimator="mean",
    errorbar="sd",
    linewidth=0.5,
    capsize=0.15,
    ax=ax,
)

sb.stripplot(
    data=sum_df,
    x="Virus_extended",
    y="percentage_csf",
    # y="percentage_csf",
    hue="catpat",
    linewidth=0.5,
    jitter=0.25,
    edgecolor="black",
    dodge=True,
    palette=catpat_palette,
    size=7,
    ax=ax,
)

# Remove duplicate legends
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:], labels[2:], title="")

ax.set_ylim(-1, 110)
ax.set_ylabel("% of cells in CSF")
ax.set_xlabel("")
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/44_VirusSpecific_in_CSF.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
## Report key numbers
sum_df[sum_df["Virus_extended"] == "EBV"].groupby("catpat")["percentage_csf"].mean()

In [ ]:
## Report key numbers
sum_df[sum_df["Virus_extended"] == "CMV"].groupby("catpat")["percentage_csf"].mean()

In [ ]:
## Report key numbers
sum_df[sum_df["Virus_extended"] == "FLU-A"].groupby("catpat")["percentage_csf"].mean()

## Joe Sabatino's metric - (%) / (%)

In [ ]:
## Filter
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]

#Save total number of PB and CSF cells
total_counts = obs_df.groupby(["donor", "catpat", "Compartment"], observed=True).size().reset_index(name="total")
total_counts = total_counts[["donor", "Compartment", "total"]].pivot(index="donor", columns="Compartment", values="total").reset_index()
total_counts = total_counts.rename(columns={"PB":"PB_total", "CSF":"CSF_total"})
PB_total = total_counts[["donor", "PB_total"]]
CSF_total = total_counts[["donor", "CSF_total"]]

#Filter
obs_df = obs_df[obs_df["total_clone_count"] > 1]
obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "FLU-A", "EBV"])]

#Drop unused categories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] =obs_df[col].cat.remove_unused_categories()

## Count and pivot to get counts per clone in a wide format
count_df = obs_df.groupby(["clone_handle", "Compartment"]).size().reset_index(name="count")
count_df = count_df.pivot(index="clone_handle", columns="Compartment", values="count")
count_df = count_df.reset_index()

## annotate
d_ann = obs_df[["donor", "clone_handle"]].drop_duplicates()
c_ann = obs_df[["donor", "catpat"]].drop_duplicates()
count_df = count_df.merge(d_ann, how="left").merge(c_ann, how="left").merge(PB_total, how="left", on="donor").merge(CSF_total, how="left",  on="donor")

## calculate percentages
count_df["CSF_pct"] = 100*count_df["CSF"] / count_df["CSF_total"]
count_df["PB_pct"] = 100*count_df["PB"] / count_df["PB_total"]

## Fold change (%) / (%)
count_df["FC_CSF"] = count_df["CSF_pct"] / count_df["PB_pct"] 

## annotate
V_ann = obs_df[["clone_handle", "Virus_extended"]].drop_duplicates()
count_df = count_df.merge(V_ann, how="left")

### %%%% Bad estimate because it doesn't deal with 0 values very well %%%% 
## I really don't see a good way of providing an adequate estimation of %PB values, if it isn't detected in PB
#### And vice versa..

## subset on clones that are shared
count_df = count_df[(count_df["CSF"] > 0) & (count_df["PB"] > 0)]

## make categories / order
count_df["catpat"] = pd.Categorical(count_df["catpat"], ordered=True, categories=["NIC", "MS"])
count_df["Virus_extended"] = pd.Categorical(count_df["Virus_extended"], ordered=True, categories=["CMV", "FLU-A", "EBV"])

## Get a mean % per donor
count_df["mean_FC_CSF"] = count_df.groupby(["donor", "Virus_extended"], observed=True)["FC_CSF"].transform("mean")
plot_df = count_df[["donor", "catpat", "Virus_extended", "mean_FC_CSF"]].drop_duplicates()

## raw data
count_df[count_df["Virus_extended"] == "EBV"].sort_values(["catpat","mean_FC_CSF"], ascending=False)
# extra_df = count_df[count_df["Virus_extended"] == "EBV"][["donor", "catpat", "Virus_extended", "CSF_total"]].drop_duplicates()
# extra_df = count_df[["donor", "catpat", "Virus_extended", "CSF_total"]].drop_duplicates()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

sb.barplot(
    data=plot_df,
    x="Virus_extended",
    # y = "CSF_total", 
    y="mean_FC_CSF",
    # y="percentage_csf",
    hue="catpat",
    palette=white_palette,
    log_scale=True,
    edgecolor="black",
    estimator="mean",
    errorbar="sd",
    linewidth=0.5,
    capsize=0.15,
    ax=ax,
)

sb.stripplot(
    data=plot_df,
    x="Virus_extended",
    # y = "CSF_total", 
    y="mean_FC_CSF",
    # y="percentage_csf",
    hue="catpat",
    linewidth=0.5,
    log_scale=True,
    jitter=0.25,
    edgecolor="black",
    dodge=True,
    palette=catpat_palette,
    size=7,
    ax=ax,
)

# Remove duplicate legends
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[2:], labels[2:], title="")

# ax.set_ylim(-1, 110)
ax.set_ylabel("Mean CSF(%) / PB(%) per donor\nFor shared clones only")
# ax.set_ylabel("Total number of CD8 T cells in CSF per donor")
ax.set_xlabel("")
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/44_VirusSpecific_in_CSF.pdf", format="pdf", dpi=400)
plt.show()

# Plot 2: UMAPs
## All antigens minus cancer

In [ ]:
mtemp = mdata.copy()
mu.pp.filter_obs(mtemp["gex"], "Virus_extended", lambda x: x != "Cancer") 

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )


In [ ]:
mtemp["gex"].obs["Antigen_extended"].value_counts()

In [ ]:
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

p1 = sc.pl.umap(
    mtemp["gex"], 
    color=["Antigen_extended"], 
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
p1.savefig("../../05_plots/13_final_plots/52_AgExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

## Virus species

In [ ]:
# Define virus colors
virus_palette = {
    "5-OP-RU": "#000000",      # orange
    "CMV": "#4C72B0",          # blue
    "Cancer": "#D7BCA6",       # light gray
    "EBV": "#DD8452",          # orange-brown
    "FLU-A": "#55A868",        # violet
    "HAdV-C": "#D62728",       # red
    "HHV-1": "#9467BD",        # purple
    "HIV": "#FF9DA6",          # pink
    "HPV": "#9D755D",          # brown
    "SARS-COV-2": "#BCBD22",   # olive
    "VZV": "#17BECF",          # cyan
    "YFV": "#693838"           # gray
}


In [ ]:
mtemp = mdata.copy()
mu.pp.filter_obs(mtemp["gex"], "Virus_extended", lambda x: x != "Cancer") 

# Remove unused categories
for col in mtemp["gex"].obs.select_dtypes(include="category").columns:
    mtemp["gex"].obs[col] = (
        mtemp["gex"].obs[col].cat.remove_unused_categories()
    )


In [ ]:
mtemp["gex"].obs["Virus_extended"].value_counts()

In [ ]:
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

p1 = sc.pl.umap(
    mtemp["gex"], 
    color=["Virus_extended"], 
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    palette=virus_palette,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

In [ ]:
mtemp["gex"].obs["vir_MS"] = np.where(mtemp["gex"].obs["catpat"] == "MS", mtemp["gex"].obs["Virus_extended"], np.nan) 

In [ ]:
print(len(mtemp["gex"].obs[~mtemp["gex"].obs["vir_MS"].isna()]["clone_handle"].drop_duplicates()))

In [ ]:
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

p1 = sc.pl.umap(
    mtemp["gex"], 
    color=["vir_MS"], 
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    palette=virus_palette,
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MSonly_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

In [ ]:
mtemp["gex"].obs["vir_NIC"] = np.where(mtemp["gex"].obs["catpat"] == "NIC", mtemp["gex"].obs["Virus_extended"], np.nan) 
print(len(mtemp["gex"].obs[~mtemp["gex"].obs["vir_NIC"].isna()]["clone_handle"].drop_duplicates()))

In [ ]:
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

p1 = sc.pl.umap(
    mtemp["gex"], 
    color=["vir_NIC"], 
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    palette=virus_palette,
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_NIConly_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

# Plot 3: Stacked bargraphs with L3_ann_final (proportion of cells)

In [ ]:
# Extract L3 annotation colors from AnnData
cats = list(mdata["gex"].obs["L3_ann"].cat.categories)

l3_palette = dict(
    zip(
        mdata["gex"].uns["L3_ann_colors"],
        mdata["gex"].uns["L3_ann_colors"]
    )
)

# The colors in .uns correspond to the categories in the original AnnData
l3_palette = dict(
    zip(
        mdata["gex"].obs["L3_ann"].cat.categories,
        mdata["gex"].uns["L3_ann_colors"]
    )
)

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"].isin(["MS", "NIC"])]
obs_df = obs_df[obs_df["Virus_extended"] != "Cancer"]
obs_df = obs_df[~obs_df["Virus_extended"].isna()]
obs_df = obs_df[obs_df["Compartment"] == "CSF"]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
   obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Count cells (include clone_handle here if counting per clone)
count_df = obs_df[["Virus_extended", "L3_ann"]].value_counts().reset_index()
count_df["total_cells"] = count_df.groupby(["Virus_extended"], observed=True)["count"].transform("sum")
count_df["percent_cells"] = 100*count_df["count"] / count_df["total_cells"]

## Exclude programs with less than 10 cells
count_df.groupby("Virus_extended")["count"].sum()

In [ ]:
## Exclude virus-specificities with too few cells
count_df = count_df[count_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "HHV-1", "SARS-COV-2"])]

## Manual cluster palette: 
man_l3_pal = {
    "TCSF (GZMKhi HLA-DRB1hi)":  "#6baed6",  # light blue
    "TCSF (GZMKhi Ribohi)":  "#3182bd",  # medium blue
    "TCSF (GZMKhi IL7Rhi)": "#9e9ac8",  # light purple
    "TCSF (CXCR6hi XCL1hi)": "#756bb1",  # purple
    "TCSF (TCF7hi LTBhi)": "#1f4fbf",  # dark blue
    "TEM (GZMKhi IFNlo)":  "#fdae6b",
}

# Pivot to wide format
plot_df = (
    count_df
    .pivot(
        index=["Virus_extended"],
        columns="L3_ann",
        values="percent_cells"
    )
    .fillna(0)
)

# Optional ordering
virus_order = [
    "5-OP-RU",
    "YFV",
    "HIV",
    "HPV",
    "CMV",
    "FLU-A",
    "HHV-1",
    "SARS-COV-2",
    "HAdV-C",
    "EBV",
    "VZV",

]

# Reorder x-axis
plot_df = plot_df.reindex(
    [v for v in virus_order if v in plot_df.index]
)

# Plot
fig, ax = plt.subplots(figsize=(6,4))

plot_df.plot(
    kind="bar",
    stacked=True,
    color=[man_l3_pal[c] for c in plot_df.columns],
    width=0.8,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("% of cells")
ax.set_xlabel("")
ax.set_title("Percentage of cells")
ax.set_ylim(0, 100)

sb.despine()
ax.grid(False)

ax.legend(
    title="L3 annotation",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/53_Extended_CSF_phenotypes.pdf", format="pdf", dpi=400)

## Stacked bargrahps in CSF - NIC vs MS

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"].isin(["MS", "NIC"])]
obs_df = obs_df[obs_df["Virus_extended"] != "Cancer"]
obs_df = obs_df[~obs_df["Virus_extended"].isna()]
obs_df = obs_df[obs_df["Compartment"] == "CSF"]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
   obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Count cells (include clone_handle here if counting per clone)
count_df = obs_df[["Virus_extended","catpat", "L3_ann"]].value_counts().reset_index()
count_df["total_cells"] = count_df.groupby(["Virus_extended", "catpat"], observed=True)["count"].transform("sum")
count_df["percent_cells"] = 100*count_df["count"] / count_df["total_cells"]

## Exclude programs with less than 5 cells in any column
count_df.groupby(["Virus_extended", "catpat"])["count"].sum()

In [ ]:
## Exclude virus-specificities with too few cells
count_df = count_df[count_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A"])]

## catpat ext
count_df["catpat_ext"] = count_df["Virus_extended"].astype(str) + "-" + count_df["catpat"].astype(str) 

# Pivot to wide format
plot_df = (
    count_df
    .pivot(
        index=["catpat_ext"],
        columns="L3_ann",
        values="percent_cells"
    )
    .fillna(0)
)

# Optional ordering
virus_order2 = [
    "CMV-NIC",
    "CMV-MS",
    "FLU-A-NIC",
    "FLU-A-MS",
    "EBV-NIC",
    "EBV-MS",
]

# Reorder x-axis
plot_df = plot_df.reindex(
    [v for v in virus_order2 if v in plot_df.index]
)

# Plot
fig, ax = plt.subplots(figsize=(6,4))

plot_df.plot(
    kind="bar",
    stacked=True,
    color=[man_l3_pal[c] for c in plot_df.columns],
    width=0.8,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("% of cells")
ax.set_xlabel("")
ax.set_title("Among CSF clusters")
ax.set_ylim(0, 100)

sb.despine()
ax.grid(False)

ax.legend(
    title="L3 annotation",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/53_Extended_NIC&MS_CSF_phenotypes.pdf", format="pdf", dpi=400)

## Stacked bargrahps in PB - NIC vs MS

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"].isin(["MS", "NIC"])]
obs_df = obs_df[obs_df["Virus_extended"] != "Cancer"]
obs_df = obs_df[~obs_df["Virus_extended"].isna()]
obs_df = obs_df[obs_df["Compartment"] == "PB"]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
   obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

# Count cells (include clone_handle here if counting per clone)
count_df = obs_df[["Virus_extended","catpat", "L3_ann"]].value_counts().reset_index()
count_df["total_cells"] = count_df.groupby(["Virus_extended", "catpat"], observed=True)["count"].transform("sum")
count_df["percent_cells"] = 100*count_df["count"] / count_df["total_cells"]

## Exclude programs with less than 5 cells in any column
count_df.groupby(["Virus_extended", "catpat"])["count"].sum()

In [ ]:
## Exclude virus-specificities with too few cells
count_df = count_df[count_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A"])]

## catpat ext
count_df["catpat_ext"] = count_df["Virus_extended"].astype(str) + "-" + count_df["catpat"].astype(str) 

# Pivot to wide format
plot_df = (
    count_df
    .pivot(
        index=["catpat_ext"],
        columns="L3_ann",
        values="percent_cells"
    )
    .fillna(0)
)

# Optional ordering
virus_order2 = [
    "CMV-NIC",
    "CMV-MS",
    "FLU-A-NIC",
    "FLU-A-MS",
    "EBV-NIC",
    "EBV-MS",
]

# Reorder x-axis
plot_df = plot_df.reindex(
    [v for v in virus_order2 if v in plot_df.index]
)

# Plot
fig, ax = plt.subplots(figsize=(6,4))

plot_df.plot(
    kind="bar",
    stacked=True,
    # color=[man_l3_pal[c] for c in plot_df.columns],
    width=0.8,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("% of cells")
ax.set_xlabel("")
ax.set_title("Among PB clusters")
ax.set_ylim(0, 100)

sb.despine()
ax.grid(False)

ax.legend(
    title="L3 annotation",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/53_Extended_NIC&MSPB_phenotypes.pdf", format="pdf", dpi=400)

# Plot 4: % in CSF of EBV-specific clones

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Get hyperexpanded CD8 T cells in PB
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
mdata_pb = mdata.copy()
mu.pp.filter_obs(mdata_pb["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"]))) 
#Drop unusedcategories
for col in mdata_pb["gex"].obs.select_dtypes(include="category").columns:
    mdata_pb["gex"].obs[col] = mdata_pb["gex"].obs[col].cat.remove_unused_categories()

## Get a temprorary dataframe of all donor and catpat combinations
temp_donor = mdata_pb["gex"].obs[mdata_pb["gex"].obs["catpat"]== "NIC"][["donor", "catpat", "Compartment"]].drop_duplicates().copy()

## Continue filtering
mu.pp.filter_obs(mdata_pb["gex"], "Virus_extended", lambda x: (x == "EBV")) 
#Drop unusedcategories
for col in mdata_pb["gex"].obs.select_dtypes(include="category").columns:
    mdata_pb["gex"].obs[col] = mdata_pb["gex"].obs[col].cat.remove_unused_categories()

## Greate the obs_df dataframe
obs_df = mdata_pb["gex"].obs[["catpat", "donor", "Compartment", "clone_handle", "type_x_clone_count", "total_clone_count"]].drop_duplicates().copy()

# Subset on patient group and expansion threshold
obs_df = obs_df[~obs_df["clone_handle"].isna()]
obs_df = obs_df[obs_df["catpat"] == "NIC"]
obs_df = obs_df[obs_df["total_clone_count"] > 15]

# #Turn all categories into strings
for col in obs_df.select_dtypes(["category"]):
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Calculate a sum of type_x_clone_count
obs_df = obs_df.groupby(["catpat", "donor", "Compartment"], observed=True, as_index=False)["type_x_clone_count"].sum()

## Left_join to preserve categories and fill with 0
obs_df = temp_donor.merge(obs_df, how = "left")
obs_df["type_x_clone_count"] = obs_df["type_x_clone_count"].fillna(0)

# Pivot to determine CSF/PB presence
obs_df = obs_df.pivot_table(
    index=["donor"],
    columns="Compartment",
    values="type_x_clone_count",
    fill_value=0
)

# Calculate %CSF
obs_df = obs_df.reset_index()
obs_df["total"] = obs_df["CSF"] + obs_df["PB"]
obs_df["CSF_pct"] = 100*obs_df["CSF"] / obs_df["total"]
obs_df["CSF_pct"] = obs_df["CSF_pct"].fillna(0)
obs_df

In [ ]:
def expansion_sweep_df_donor3(sdata_temp, catpat="MS", max_expansion=25):

    all_results = []

    for x in range(0, max_expansion + 1):

        # Get relevant columns
        obs_df = sdata_temp["gex"].obs[
             ["catpat", "donor", "Compartment", "clone_handle", "type_x_clone_count", "total_clone_count"] ## By dropping duplicates each clone will only be present once. 
        ].drop_duplicates().copy()

        # Get a temp_donor df with all the categories you want to preserve later
        temp_donor = obs_df[obs_df["catpat"]==catpat][["donor", "catpat", "Compartment"]].drop_duplicates()
        
        #Drop unusedcategories
        for col in temp_donor.select_dtypes(include="category").columns:
            temp_donor[col] = temp_donor[col].cat.remove_unused_categories()
        
        # Subset on patient group and expansion threshold
        obs_df = obs_df[~obs_df["clone_handle"].isna()]  #Removes those cells without a clone
        obs_df = obs_df[obs_df["catpat"] == catpat]
        obs_df = obs_df[obs_df["total_clone_count"] > x]

        #Turn all categories into strings
        for col in obs_df.select_dtypes(["category"]):
            obs_df[col] = obs_df[col].cat.remove_unused_categories()

        ## Calculate a sum of type_x_clone_count
        obs_df = obs_df.groupby(["catpat", "donor", "Compartment"], observed=True, as_index=False)["type_x_clone_count"].sum()

        ## Left_join to preserve categories and fill with 0
        obs_df = temp_donor.merge(obs_df, how = "left")
        obs_df["type_x_clone_count"] = obs_df["type_x_clone_count"].fillna(0)

        # Pivot to determine CSF/PB presence
        obs_df = obs_df.pivot_table(
            index=["donor"],
            columns="Compartment",
            values="type_x_clone_count",
            fill_value=0
        )
        

        # Calculate %CSF
        obs_df = obs_df.reset_index()
        obs_df["total"] = obs_df["CSF"] + obs_df["PB"]
        obs_df["CSF_pct"] = 100*obs_df["CSF"] / obs_df["total"]
        obs_df["CSF_pct"] = obs_df["CSF_pct"].fillna(0)

        #Shift 0 values to prepare for log
        obs_df["CSF_pct"] = obs_df["CSF_pct"]+1

        # Add expansion threshold
        obs_df["expansion_threshold"] = x

        # Add patient category
        obs_df["catpat"] = catpat

        all_results.append(obs_df)

    return pd.concat(all_results, ignore_index=True)

In [ ]:
## Make the dataframes
df_MS = expansion_sweep_df_donor3(mdata_pb, "MS")
df_ctrl = expansion_sweep_df_donor3(mdata_pb, "NIC")

## Concatenate
out_concat = pd.concat([df_MS, df_ctrl])

In [ ]:
plot_df = out_concat[out_concat["expansion_threshold"].isin([0, 5, 10, 15, 20])]

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
sb.boxplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("% of cells in CSF per donor") #% of cells in CSF\nnorm. to clone sizes
ax.set_title("Of EBV-specific cells")

sb.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/13_boxplots_catexp_thresholds_percentage_cells.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = [0, 5, 10, 15, 20]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = plot_df[plot_df["expansion_threshold"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["CSF_pct"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["CSF_pct"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

# Plot 5: Stacked bar graphs - per donor

In [ ]:
## Virus/multimer annotation
obs_df = mdata["gex"].obs.copy()

# Focus on multimer annotated cells
obs_df = obs_df[~obs_df["Virus_extended"].isna()].copy()
obs_df = obs_df[obs_df["catpat"].isin(["NIC", "MS"])]

# Remove unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = (
        obs_df[col].cat.remove_unused_categories()
    )

mult_ann = obs_df[["clone_handle", "Virus_extended"]].drop_duplicates()

# Original hyperexpanded clones
clone_table = pd.read_csv("../../06_csv_outputs/Clones/2026_7_CD8_hyperexpanded_clone_handles_and_composition_table_across_compartments.csv")

# Join
plot_df = clone_table.merge(mult_ann, how="left")

# Add no epiope
plot_df["Virus_extended"] = np.where(plot_df["Virus_extended"].isna(), "Not annotated", plot_df["Virus_extended"])

# Redefine the donor column
plot_df["donor"] = plot_df["clone_handle"].str.split('-').str[0]

# count how many we deorphanized per donor
count_df = plot_df.groupby(["donor", "Virus_extended"],observed=True).size().reset_index(name="count")

#Colors
virus_colors = {
    "EBV": "#DD8452",
    "CMV": "#4C72B0",
    "FLU-A": "#55A868",
    "VZV": "#17BECF",
    "Not annotated" : "#D3D3D3"
}


# Pivot to wide format
plot_df = (
    count_df
    .pivot(
        index="donor",
        columns="Virus_extended",
        values="count"
    )
    .fillna(0)
)

# Reorder columns according to dictionary order
plot_df = plot_df[
    [v for v in virus_colors.keys() if v in plot_df.columns]
]

# Ensure colors follow the plotted column order
colors = [
    virus_colors[v]
    for v in plot_df.columns
]


# Apply desired donor order
ordered_donor_list = ["Pt1","Pt8", "Pt18", "Pt19", "Pt21",  "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
plot_df = plot_df.reindex(
    [d for d in ordered_donor_list if d in plot_df.index]
)

# Plot
fig, ax = plt.subplots(figsize=(6, 4))

plot_df.plot(
    kind="bar",
    stacked=True,
    color=colors,
    width=0.8,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("Number of clones")
ax.set_xlabel("Donor")

sb.despine()
ax.grid(False)

ax.legend(
    title="Virus",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/42_Deorph_hypexp_Ext.pdf", format="pdf", dpi=400)
plt.show()

# Plot 6: Stacked bar graphs - PB/CSF

In [ ]:
# Keep catpat information
mult_ann = mult_ann = obs_df[["clone_handle", "Virus_extended"]].drop_duplicates()

# Join
plot_df = clone_table.merge(
    mult_ann,
    how="left"
)

# Drop unannotated
plot_df2 = plot_df.dropna(subset="Virus_extended").copy()

# Sort by CSF contribution
plot_df2 = plot_df2.sort_values("CSF.pct", ascending=False)

# Select relevant columns
plot_long = plot_df2.melt(
    id_vars=["clone_handle", "Virus_extended", "catpat"],
    value_vars=["CSF.pct", "PB.pct"],
    var_name="Compartment",
    value_name="Percent"
)

# Pivot for stacked barplot
plot_pivot = (
    plot_long
    .pivot(
        index="clone_handle",
        columns="Compartment",
        values="Percent"
    )
    .fillna(0)
)

plot_pivot = plot_pivot[["CSF.pct", "PB.pct"]]

# Preserve CSF sorting
clone_order = plot_df2["clone_handle"].drop_duplicates()
plot_pivot = plot_pivot.reindex(clone_order)


# Colors
virus_colors = {
    "EBV": "#DD8452",
    "CMV": "#4C72B0",
    "FLU-A": "#55A868",
    "HHV-1": "#9467BD",
    "HHV-2": "#8E1EF4",
    "VZV": "#17BECF",
    "HMPV": "#E377C2",
    "SARS-COV-2": "#BCBD22",
    "HAdV-C": "#D62728",
    "NWV": "#7F7F7F"
}

catpat_colors = {
    "NIC": "#FFFFFF",
    "MS": "#000000"
}


# Match annotations to clone order
annotation_df = (
    plot_df2
    .drop_duplicates("clone_handle")
    .set_index("clone_handle")
    .loc[clone_order]
)

virus_track = annotation_df["Virus_extended"]
catpat_track = annotation_df["catpat"]

virus_rgb = [
    plt.matplotlib.colors.to_rgb(virus_colors[v])
    for v in virus_track
]

catpat_rgb = [
    plt.matplotlib.colors.to_rgb(catpat_colors[v])
    for v in catpat_track
]


# Figure with two annotation tracks
fig, (ax, ax_virus, ax_catpat) = plt.subplots(
    nrows=3,
    figsize=(6, 6),
    gridspec_kw={
        "height_ratios": [8, 0.4, 0.4]
    },
    sharex=True
)


# --------------------
# Main stacked barplot
# --------------------
plot_pivot.plot(
    kind="bar",
    stacked=True,
    color=["#4B8AC1", "#C06A6A"],
    width=1,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("Compartment (%)")
ax.set_xlabel("")
ax.set_ylim(0, 100)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    handles=[
        Patch(facecolor="#4B8AC1", label="CSF"),
        Patch(facecolor="#C06A6A", label="PB"),
    ],
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

ax.grid(False)


# --------------------
# Virus annotation strip
# --------------------
ax_virus.imshow(
    np.array([virus_rgb]),
    aspect="auto",
    interpolation="nearest"
)

ax_virus.set_yticks([])
ax_virus.set_xticks([])


# --------------------
# Cohort annotation strip
# --------------------
ax_catpat.imshow(
    np.array([catpat_rgb]),
    aspect="auto",
    interpolation="nearest"
)

ax_catpat.set_yticks([])
ax_catpat.set_xticks([])


# Remove annotation axes borders
for annotation_ax in [ax_virus, ax_catpat]:
    for spine in annotation_ax.spines.values():
        spine.set_visible(False)


# Legends
virus_legend = [
    Patch(facecolor=color, label=virus)
    for virus, color in virus_colors.items()
    if virus in virus_track.values
]

ax_virus.legend(
    handles=virus_legend,
    title="Virus specificity",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

catpat_legend = [
    Patch(facecolor=color, label=group)
    for group, color in catpat_colors.items()
    if group in catpat_track.values
]

ax_catpat.legend(
    handles=catpat_legend,
    title="Cohort",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)


plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/40_hyperexpanded_clone_deorphanization_ext.pdf")

# Plot 7: Stacked bar graphs - MS vs NIC

## CSF-biased clones

In [ ]:
obs_df = mdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Filter by the CSF-biased clones
# CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
# biased_clones = set(CSFbiased)
# obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
biased_clones = set(CSFbiased)
obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]
print(len(obs_df["donor"].drop_duplicates()))

## Summarize
sum_df = obs_df[["catpat","clone_handle", "Virus_extended"]].drop_duplicates()
sum_df["Virus_extended"] = np.where(sum_df["Virus_extended"].astype(str) == "nan", "Unknown", sum_df["Virus_extended"])
sum_df["Virus_extended"] = pd.Categorical(sum_df["Virus_extended"], categories=["CMV", "EBV", "FLU-A", "HHV-1", "VZV", "Unknown"], ordered=True)
print(sum_df[["catpat"]].value_counts())
sum_df = sum_df.groupby(["Virus_extended","catpat"])["Virus_extended"].size().reset_index(name="count")

# ## Percentage
sum_df = sum_df.pivot(index="catpat", columns="Virus_extended", values="count")
sum_df["total"] = sum_df.sum(axis=1)
sum_df["EBV"] = sum_df["EBV"] / sum_df["total"]*100
sum_df["CMV"] = sum_df["CMV"] / sum_df["total"]*100
sum_df["FLU-A"] = sum_df["FLU-A"] / sum_df["total"]*100
sum_df["Unknown"] = sum_df["Unknown"] / sum_df["total"]*100

# Long format
plot_df = (
    sum_df[["CMV", "EBV", "FLU-A",  "HHV-1", "VZV", "Unknown"]]
    .reset_index()
    .melt(
        id_vars="catpat",
        var_name="Virus",
        value_name="Percent"
    )
)

plot_df["catpat"] = pd.Categorical(
    plot_df["catpat"],
    categories=["NIC", "MS"],
    ordered=True,
)


virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "HHV-1" : "#9467BD", 
    "VZV" : "#17BECF", 
    "Unknown" : "#F5F5F5"
}

fig, ax = plt.subplots(figsize=(4, 4))

order = ["NIC", "MS"]
bottom = np.zeros(len(order))

for virus in ["CMV", "EBV", "FLU-A", "HHV-1", "VZV", "Unknown"]:
    vals = (
        plot_df.loc[plot_df["Virus"] == virus]
        .set_index("catpat")
        .loc[order, "Percent"]
        .values
    )

    ax.bar(
        order,           # categorical x values
        vals,
        bottom=bottom,
        color=virus_palette[virus],
        label=virus,
        width=0.8,
    )

    bottom += vals

ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 100)
ax.legend(title="Virus", bbox_to_anchor=(1.05, 1))
sb.despine()

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/35_CEF_specificity_among_CSFbiased.pdf", format="pdf", dpi=400)
plt.show()

## PB-biased clones

In [ ]:
obs_df = mdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Filter by the CSF-biased clones
# CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
# biased_clones = set(CSFbiased)
# obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

PBbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_PB14_clonehandles_ForCSF.csv")["0"].tolist()
biased_clones = set(PBbiased)
obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

## Summarize
sum_df = obs_df[["catpat","clone_handle", "Virus_extended"]].drop_duplicates()
sum_df["Virus_extended"] = np.where(sum_df["Virus_extended"].astype(str) == "nan", "Unknown", sum_df["Virus_extended"])
print(sum_df[["catpat"]].value_counts())
sum_df["Virus_extended"] = pd.Categorical(sum_df["Virus_extended"], categories=["CMV", "EBV", "FLU-A", "HHV-1", "VZV", "Unknown"], ordered=True)
sum_df = sum_df.groupby(["Virus_extended","catpat"])["Virus_extended"].size().reset_index(name="count")

# ## Percentage
sum_df = sum_df.pivot(index="catpat", columns="Virus_extended", values="count")
sum_df["total"] = sum_df.sum(axis=1)
sum_df["EBV"] = sum_df["EBV"] / sum_df["total"]*100
sum_df["CMV"] = sum_df["CMV"] / sum_df["total"]*100
sum_df["FLU-A"] = sum_df["FLU-A"] / sum_df["total"]*100
sum_df["Unknown"] = sum_df["Unknown"] / sum_df["total"]*100

# Long format
plot_df = (
    sum_df[["CMV", "EBV", "FLU-A",  "HHV-1", "VZV", "Unknown"]]
    .reset_index()
    .melt(
        id_vars="catpat",
        var_name="Virus",
        value_name="Percent"
    )
)

plot_df["catpat"] = pd.Categorical(
    plot_df["catpat"],
    categories=["NIC", "MS"],
    ordered=True,
)


virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "HHV-1" : "#9467BD", 
    "VZV" : "#17BECF", 
    "Unknown" : "#F5F5F5"
}

fig, ax = plt.subplots(figsize=(4, 4))

order = ["NIC", "MS"]
bottom = np.zeros(len(order))

for virus in ["CMV", "EBV", "FLU-A", "HHV-1", "VZV", "Unknown"]:
    vals = (
        plot_df.loc[plot_df["Virus"] == virus]
        .set_index("catpat")
        .loc[order, "Percent"]
        .values
    )

    ax.bar(
        order,           # categorical x values
        vals,
        bottom=bottom,
        color=virus_palette[virus],
        label=virus,
        width=0.8,
    )

    bottom += vals

ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 100)
ax.legend(title="Virus", bbox_to_anchor=(1.05, 1))
sb.despine()

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/35_CEF_specificity_among_CSFbiased.pdf", format="pdf", dpi=400)
plt.show()

# Plot 8: B-LCL reactive clones in the multimer sort data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_preprocessed_annotated_intgr.h5mu")

#Group annotation
patient_list = [ "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1","Pt8", "Pt18", "Pt19", "Pt21"]

#Match the lists defined above to define "RRMS" and "Control"
sdata["gex"].obs["catpat"] = pd.Categorical(np.where(sdata["gex"].obs["donor"].isin(patient_list), "MS", np.where(sdata["gex"].obs["donor"].isin(control_list), "NIC", "IC")))

## Correct multimer annotations
## UL28/29 is from the FRCPRRFCF epitope. Antigen was never made NA if CMV seronegative or FRC positive. Do this now
sdata["gex"].obs["Antigen"] = np.where(sdata["gex"].obs["Sequence"].isna(), sdata["gex"].obs["Sequence"], sdata["gex"].obs["Antigen"])

In [ ]:
#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## Get obs
obs_df = sdata["gex"].obs.copy()
obs_df =obs_df[obs_df["catpat"]!="IC"]
obs_df =obs_df[obs_df["donor"].isin(["Pt21", "Pt20", "Pt19", "Pt18", "Pt17", "Pt14"])]
obs_df = obs_df[~obs_df["Antigen"].isna()]

# Filter to multimer positive clones
exp = sdata["gex"].obs[~sdata["gex"].obs["Sequence"].isna()]["clone_handle"].drop_duplicates().tolist()
obs_df = obs_df[obs_df["clone_handle"].isin(exp)]

#Get this order
antigen_order = obs_df[["Antigen", "Virus"]].value_counts().reset_index().sort_values(["Virus", "count"])["Antigen"].tolist()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## Count
obs_df = obs_df[["clone_handle", "Antigen", "LCL_reactive"]].drop_duplicates()
obs_df["Antigen"] = pd.Categorical(obs_df["Antigen"]) 
obs_df = obs_df.groupby(["Antigen", "LCL_reactive"]).size().reset_index(name="count").sort_values("Antigen")

## pivot
plot_df = obs_df.pivot(index="Antigen", columns="LCL_reactive", values="count").reset_index()
plot_df["Total"] = plot_df["LCL-reactive"]+plot_df["Non-reactive"]
print(plot_df.sort_values("Total", ascending=False))

## Use stacked bargraphs to show how many "LCL-reactive" and "Non-reactive" counts for each Antigen were counted: 
# Set Antigen as index
plot_df["Antigen"] = plot_df["Antigen"].astype(str)
plot_df = plot_df.fillna(0)

plot_df = plot_df.set_index("Antigen")
plot_df = plot_df.reindex([a for a in antigen_order if a in plot_df.index])

# Define colors
LCL_palette = {
    "LCL-reactive": "#8B0000",
    "Non-reactive": "#B0B0B0",
}

fig, ax = plt.subplots(figsize=(10, 4))

plot_df[["LCL-reactive", "Non-reactive"]].plot(
    kind="bar",
    stacked=True,
    color=[LCL_palette["LCL-reactive"], LCL_palette["Non-reactive"]],
    width=0.8,
    edgecolor="black",
    linewidth=0.3,
    ax=ax
)

ax.set_ylabel("Number of clones")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=90)

ax.legend(
    title="",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

# ax.set_ylim(-1, 150)

sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/43_LCL_reactive_multimer_positive_clones.pdf", dpi=400, format="pdf")
plt.show()



# Plot 9: LCL-reactive clones among CSF-biased and PB-biased clones

In [ ]:
#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## Get obs from the CD8 data
obs_df = mdata["gex"].obs.copy()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12"])]
CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
PBbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_PB14_clonehandles_ForCSF.csv")["0"].tolist()
obs_df["catbiased"] = np.where(obs_df["clone_handle"].isin(set(CSFbiased)), "CSF-biased", 
                               np.where(obs_df["clone_handle"].isin(set(PBbiased)), "PB-biased", "Unselected"))
obs_df = obs_df[obs_df["catbiased"].isin(["CSF-biased", "PB-biased"])]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] =obs_df[col].cat.remove_unused_categories()

## Summarize
sum_df = obs_df[["catpat","clone_handle", "LCL_reactive", "catbiased"]].drop_duplicates()
sum_df = sum_df.groupby(["LCL_reactive","catpat", "catbiased"])["LCL_reactive"].size().reset_index(name="count")
sum_df = sum_df.pivot(index=["catpat", "catbiased"], columns="LCL_reactive", values="count").reset_index()
print(sum_df)

## Convert to percentage
sum_df = sum_df.set_index(["catpat", "catbiased"])
sum_df["total"] = sum_df.sum(axis=1)
sum_df["LCL_pct"] = sum_df["LCL-reactive"] / sum_df["total"]*100
sum_df["NR"] = sum_df["Non-reactive"] / sum_df["total"]*100

# colors
compartment_palette = {
    "NIC": "#808080", #"#4B8AC1",   # blue
    "MS":  "#000000" #"#C06A6A"    # red
}

# Plot
fig, ax = plt.subplots(figsize=(5.5,5))
sb.barplot(data=sum_df, x="catbiased", y="LCL_pct", hue="catpat", hue_order=["NIC", "MS"], palette=compartment_palette, ax=ax)

plt.xticks(rotation=0)
plt.ylabel("% LCL-reactive CD8 T-cell clones")
plt.xlabel(" ")
plt.ylim(0,40)
plt.title("NIC (n=3), MS (n=3)")

# Move legend to the right
ax.legend(title="Compartment", bbox_to_anchor=(1.05, 1), loc="upper left")

sb.despine()
ax.grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/35_LCLreact_MSvsNIC.pdf")
plt.show()

In [ ]:
sum_df.reset_index()

In [ ]:
from scipy.stats import fisher_exact

for catpat in ["MS", "NIC"]:
    df = sum_df.reset_index()[sum_df.reset_index()["catpat"] == catpat]

    contingency = (
        df.set_index("catbiased")[["LCL-reactive", "Non-reactive"]]
    )

    odds_ratio, p_value = fisher_exact(contingency)

    print(f"\n{catpat}")
    print(contingency)
    print(f"Odds ratio = {odds_ratio:.2f}")
    print(f"P-value = {p_value:.4g}")

# Plot 10: LCL react + multimer + computation annotations - how much can we explain? 

## CSF-biased

In [ ]:
obs_df = mdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Filter by the CSF-biased clones
CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv")["0"].tolist()
biased_clones = set(CSFbiased)
obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12"])]

#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## EXTEND VIRUS EXTENDED
obs_df["Virus_extended"] = np.where((obs_df["Virus_extended"].isna()) & (obs_df["LCL_reactive"] == "LCL-reactive"), "LCL-reactive", obs_df["Virus_extended"])
print(obs_df["Virus_extended"].drop_duplicates().tolist())
## Summarize
sum_df = obs_df[["catpat","clone_handle", "Virus_extended"]].drop_duplicates()
sum_df["Virus_extended"] = np.where(sum_df["Virus_extended"].astype(str) == "nan", "Unknown", sum_df["Virus_extended"])
print(sum_df[["catpat", "Virus_extended"]].value_counts())
sum_df["Virus_extended"] = pd.Categorical(sum_df["Virus_extended"], categories=["CMV", "EBV", "FLU-A", "HHV-1", "VZV", "LCL-reactive", "Unknown"], ordered=True)
sum_df = sum_df.groupby(["Virus_extended","catpat"])["Virus_extended"].size().reset_index(name="count")

# ## Percentage
sum_df = sum_df.pivot(index="catpat", columns="Virus_extended", values="count")
sum_df["total"] = sum_df.sum(axis=1)
sum_df["EBV"] = sum_df["EBV"] / sum_df["total"]*100
sum_df["CMV"] = sum_df["CMV"] / sum_df["total"]*100
sum_df["FLU-A"] = sum_df["FLU-A"] / sum_df["total"]*100
sum_df["LCL-reactive"] = sum_df["LCL-reactive"] / sum_df["total"]*100
sum_df["Unknown"] = sum_df["Unknown"] / sum_df["total"]*100

# Long format
plot_df = (
    sum_df[["CMV", "EBV", "FLU-A",  "HHV-1", "VZV", "LCL-reactive", "Unknown"]]
    .reset_index()
    .melt(
        id_vars="catpat",
        var_name="Virus",
        value_name="Percent"
    )
)

plot_df["catpat"] = pd.Categorical(
    plot_df["catpat"],
    categories=["NIC", "MS"],
    ordered=True,
)


virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "HHV-1" : "#9467BD", 
    "VZV" : "#17BECF", 
    "LCL-reactive" : "#000000", 
    "Unknown" : "#F5F5F5"
}

fig, ax = plt.subplots(figsize=(4, 4))

order = ["NIC", "MS"]
bottom = np.zeros(len(order))

for virus in ["CMV", "EBV", "FLU-A", "HHV-1", "VZV",  "LCL-reactive", "Unknown"]:
    vals = (
        plot_df.loc[plot_df["Virus"] == virus]
        .set_index("catpat")
        .loc[order, "Percent"]
        .values
    )

    ax.bar(
        order,           # categorical x values
        vals,
        bottom=bottom,
        color=virus_palette[virus],
        label=virus,
        width=0.8,
    )

    bottom += vals

ax.set_ylabel("Percentage (%)")
ax.set_title("CSF-biased clonotypes")
ax.set_ylim(0, 100)
ax.legend(title="Virus", bbox_to_anchor=(1.05, 1))
sb.despine()

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/35_CEF_specificity_among_CSFbiased.pdf", format="pdf", dpi=400)
plt.show()

## PB-biased

In [ ]:
obs_df = mdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Filter by the CSF-biased clones
CSFbiased = pd.read_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_PB14_clonehandles_ForCSF.csv")["0"].tolist()
biased_clones = set(CSFbiased)
obs_df = obs_df[obs_df["clone_handle"].isin(biased_clones)]

## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12"])]

#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## EXTEND VIRUS EXTENDED
obs_df["Virus_extended"] = np.where((obs_df["Virus_extended"].isna()) & (obs_df["LCL_reactive"] == "LCL-reactive"), "LCL-reactive", obs_df["Virus_extended"])
print(obs_df["Virus_extended"].drop_duplicates().tolist())
## Summarize
sum_df = obs_df[["catpat","clone_handle", "Virus_extended"]].drop_duplicates()
sum_df["Virus_extended"] = np.where(sum_df["Virus_extended"].astype(str) == "nan", "Unknown", sum_df["Virus_extended"])
print(sum_df[["catpat"]].value_counts())
sum_df["Virus_extended"] = pd.Categorical(sum_df["Virus_extended"], categories=["CMV", "EBV", "FLU-A", "HHV-1", "VZV", "LCL-reactive", "Unknown"], ordered=True)
sum_df = sum_df.groupby(["Virus_extended","catpat"])["Virus_extended"].size().reset_index(name="count")

# ## Percentage
sum_df = sum_df.pivot(index="catpat", columns="Virus_extended", values="count")
sum_df["total"] = sum_df.sum(axis=1)
sum_df["EBV"] = sum_df["EBV"] / sum_df["total"]*100
sum_df["CMV"] = sum_df["CMV"] / sum_df["total"]*100
sum_df["FLU-A"] = sum_df["FLU-A"] / sum_df["total"]*100
sum_df["HHV-1"] = sum_df["HHV-1"] / sum_df["total"]*100
sum_df["LCL-reactive"] = sum_df["LCL-reactive"] / sum_df["total"]*100
sum_df["Unknown"] = sum_df["Unknown"] / sum_df["total"]*100

# Long format
plot_df = (
    sum_df[["CMV", "EBV", "FLU-A",  "HHV-1", "VZV", "LCL-reactive", "Unknown"]]
    .reset_index()
    .melt(
        id_vars="catpat",
        var_name="Virus",
        value_name="Percent"
    )
)

plot_df["catpat"] = pd.Categorical(
    plot_df["catpat"],
    categories=["NIC", "MS"],
    ordered=True,
)


virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "HHV-1" : "#9467BD", 
    "VZV" : "#17BECF", 
    "LCL-reactive" : "#000000", 
    "Unknown" : "#F5F5F5"
}

fig, ax = plt.subplots(figsize=(4, 4))

order = ["NIC", "MS"]
bottom = np.zeros(len(order))

for virus in ["CMV", "EBV", "FLU-A", "HHV-1", "VZV",  "LCL-reactive", "Unknown"]:
    vals = (
        plot_df.loc[plot_df["Virus"] == virus]
        .set_index("catpat")
        .loc[order, "Percent"]
        .values
    )

    ax.bar(
        order,           # categorical x values
        vals,
        bottom=bottom,
        color=virus_palette[virus],
        label=virus,
        width=0.8,
    )

    bottom += vals

ax.set_ylabel("Percentage (%)")
ax.set_title("PB-biased clonotypes")
ax.set_ylim(0, 100)
ax.legend(title="Virus", bbox_to_anchor=(1.05, 1))
sb.despine()

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/35_CEF_specificity_among_CSFbiased.pdf", format="pdf", dpi=400)
plt.show()

## All hyperexpanded clones

In [ ]:
obs_df = mdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

## All hyperexpanded clones
obs_df = obs_df[obs_df["total_clone_count"] > 12]

## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12"])]

#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## annotate
obs_df["clone_handle2"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str) 
obs_df["LCL_reactive"] = np.where(obs_df["clone_handle2"].isin(LCL_list), "LCL-reactive", "Non-reactive")

## EXTEND VIRUS EXTENDED
obs_df["Virus_extended"] = np.where((obs_df["Virus_extended"].isna()) & (obs_df["LCL_reactive"] == "LCL-reactive"), "LCL-reactive", obs_df["Virus_extended"])
print(obs_df["Virus_extended"].drop_duplicates().tolist())
## Summarize
sum_df = obs_df[["catpat","clone_handle", "Virus_extended"]].drop_duplicates()
sum_df["Virus_extended"] = np.where(sum_df["Virus_extended"].astype(str) == "nan", "Unknown", sum_df["Virus_extended"])
print(sum_df[["catpat"]].value_counts())
sum_df["Virus_extended"] = pd.Categorical(sum_df["Virus_extended"], categories=["CMV", "EBV", "FLU-A", "HHV-1", "VZV", "LCL-reactive", "Unknown"], ordered=True)
sum_df = sum_df.groupby(["Virus_extended","catpat"])["Virus_extended"].size().reset_index(name="count")

# ## Percentage
sum_df = sum_df.pivot(index="catpat", columns="Virus_extended", values="count")
sum_df["total"] = sum_df.sum(axis=1)
sum_df["EBV"] = sum_df["EBV"] / sum_df["total"]*100
sum_df["CMV"] = sum_df["CMV"] / sum_df["total"]*100
sum_df["HHV-1"] = sum_df["HHV-1"] / sum_df["total"]*100
sum_df["FLU-A"] = sum_df["FLU-A"] / sum_df["total"]*100
sum_df["LCL-reactive"] = sum_df["LCL-reactive"] / sum_df["total"]*100
sum_df["Unknown"] = sum_df["Unknown"] / sum_df["total"]*100

# Long format
plot_df = (
    sum_df[["CMV", "EBV", "FLU-A",  "HHV-1", "VZV", "LCL-reactive", "Unknown"]]
    .reset_index()
    .melt(
        id_vars="catpat",
        var_name="Virus",
        value_name="Percent"
    )
)

plot_df["catpat"] = pd.Categorical(
    plot_df["catpat"],
    categories=["NIC", "MS"],
    ordered=True,
)


virus_palette = {
    "CMV" : "#4C72B0", 
    "EBV" : "#DD8452", 
    "FLU-A" : "#55A868", 
    "HHV-1" : "#9467BD", 
    "VZV" : "#17BECF", 
    "LCL-reactive" : "#000000", 
    "Unknown" : "#F5F5F5"
}

fig, ax = plt.subplots(figsize=(4, 4))

order = ["NIC", "MS"]
bottom = np.zeros(len(order))

for virus in ["CMV", "EBV", "FLU-A", "HHV-1", "VZV",  "LCL-reactive", "Unknown"]:
    vals = (
        plot_df.loc[plot_df["Virus"] == virus]
        .set_index("catpat")
        .loc[order, "Percent"]
        .values
    )

    ax.bar(
        order,           # categorical x values
        vals,
        bottom=bottom,
        color=virus_palette[virus],
        label=virus,
        width=0.8,
    )

    bottom += vals

ax.set_ylabel("Percentage (%)")
ax.set_title("All hyperexpanded clones")
ax.set_ylim(0, 100)
ax.legend(title="Virus", bbox_to_anchor=(1.05, 1))
sb.despine()

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/35_CEF_specificity_among_CSFbiased.pdf", format="pdf", dpi=400)
plt.show()

# Plot 11: Upset plot

In [ ]:
# See NK13 Extended virus annotations (the preceeding analysis file)

# Plot 12: Quantifications
- % of cells for each virus-specificity in CSF for NIC and MS
- % of cells for each virus-specificity in PB for NIC and MS
- Number of virus-specific cells / ml in NIC and MS

## % of CSF and PB

In [ ]:
## obs_df = mdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["Virus_extended"] != "Cancer"]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Create a temp df of cells in PB and CSF per donor
temp_df = obs_df[["donor", "catpat", "Compartment"]].value_counts().reset_index(name = "total")

## Count_df per virus specifcity
count_df = obs_df[["donor", "Compartment", "Virus_extended"]].value_counts().reset_index(name = "Virus_counts")

# Generate all possible combinations
all_combinations = pd.MultiIndex.from_product([
        obs_df["donor"].unique(),
        obs_df["Compartment"].unique(),
        obs_df["Virus_extended"].unique(),
    ],
    names=["donor", "Compartment", "Virus_extended"]
)

# Add missing combinations with count = 0
count_df = (
    count_df
    .set_index(["donor", "Compartment", "Virus_extended"])
    .reindex(all_combinations, fill_value=0)
    .reset_index()
)

## left join onto temp_df to preserve all categories
merged_df = temp_df.merge(count_df, how = "left")

## percentages
merged_df["Virus_pct"] = 100*merged_df["Virus_counts"] / merged_df["total"]

## Plot MS vs NIC for each Virus_extended
# colors
catpat_palette = {
    "NIC": "#FFFFFF", 
    "MS":  "#000000" 
}

white_palette = {
    "MS": "white",
    "NIC": "white",
}

g = sb.catplot(
    data=merged_df,
    x="Virus_extended",
    y="Virus_pct",
    hue="catpat",
        hue_order=["NIC", "MS"],   # explicitly enforce order
    col="Compartment",
    kind="box",
    palette=white_palette,
    dodge=True,
    showfliers=False,
    linewidth=0.5,
    height=4,
    aspect=1,
)

# Add individual observations
for ax, (compartment, df_comp) in zip(
    g.axes.flat,
    merged_df.groupby("Compartment", observed=True)
):
    sb.stripplot(
        data=df_comp,
        x="Virus_extended",
        y="Virus_pct",
        hue="catpat",
        hue_order=["NIC", "MS"],   # explicitly enforce order
        linewidth=0.5,
        jitter=0.25,
        edgecolor="black",
        dodge=True,
        palette=catpat_palette,
        size=6,
        ax=ax,
        legend=False,
    )

    ax.set_xlabel("")
    ax.grid(False)
    sb.despine(ax=ax)

# Axis labels
g.set_axis_labels("", "Virus frequency (%)")

# Remove the automatic legend and add one clean legend
if g._legend is not None:
    g._legend.remove()

handles = [
    plt.Line2D(
        [0], [0],
        marker="o",
        color="black",
        markerfacecolor=catpat_palette[cat],
        markeredgecolor="black",
        markersize=7,
        linestyle=""
    )
    for cat in ["NIC", "MS"]
]

g.fig.legend(
    handles,
    ["NIC", "MS"],
    title="",
    loc="upper right",
    bbox_to_anchor=(1.02, 1)
)

g.set_xticklabels(rotation=45, ha="right")

plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/XX_VirusSpecificityPct.pdf", dpi=400)
plt.show()

## Counts per ml CSF

In [ ]:
## obs_df = mdata["gex"].obs.copy()
## Filter
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["Virus_extended"] != "Cancer"]
obs_df = obs_df[obs_df["Compartment"] == "CSF"]

#Drop unusedcategories
for col in obs_df.select_dtypes(include="category").columns:
  obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Create a temp df of cells in PB and CSF per donor
temp_df = obs_df[["donor", "catpat", "Compartment"]].value_counts().reset_index(name = "total")

## Count_df per virus specifcity
count_df = obs_df[["donor", "Compartment", "Virus_extended"]].value_counts().reset_index(name = "Virus_counts")

# Generate all possible combinations
all_combinations = pd.MultiIndex.from_product([
        obs_df["donor"].unique(),
        obs_df["Compartment"].unique(),
        obs_df["Virus_extended"].unique(),
    ],
    names=["donor", "Compartment", "Virus_extended"]
)

# Add missing combinations with count = 0
count_df = (
    count_df
    .set_index(["donor", "Compartment", "Virus_extended"])
    .reindex(all_combinations, fill_value=0)
    .reset_index()
)

## left join onto temp_df to preserve all categories
merged_df = temp_df.merge(count_df, how = "left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
merged_df["csf_vol"] = merged_df["donor"].map(csf_ml)

## normalize cell_count
merged_df["cell_count_norm"] = merged_df["Virus_counts"] / merged_df["csf_vol"]

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
#             plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
sb.boxplot(
    data=merged_df,
    x="Virus_extended",
    y="cell_count_norm",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=merged_df,
    x="Virus_extended",
    y="cell_count_norm",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_xlabel("Virus-specificities")
ax.set_ylabel("CD8 T-cells in CSF/ml") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/XX_normalized_cell_count_csf_viruspecificity.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()


# Plot 13: mult vs main - rediscovered clones

In [ ]:
obs_df = mdata["gex"].obs.copy()
df = obs_df[["clone_handle", "Virus"]].drop_duplicates().value_counts().reset_index(name = "counts")
df = df[["Virus"]].value_counts().reset_index(name="counts")


# Order Virus_extendedes by counts
order = df.sort_values("counts", ascending=False)["Virus"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(4, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Clonotypes in both dataset")
plt.xlabel("")
# plt.title("Clones present in both the main\n and multimer-sort datasets")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/25_NumberOfMatchedClones_MultvMain.pdf")

# Plot 14: extended vs main - rediscovered clones

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Virus_extended"] != "Cancer"]
df = obs_df[["clone_handle", "Virus_extended"]].drop_duplicates().value_counts().reset_index(name = "counts")
df = df[["Virus_extended"]].value_counts().reset_index(name="counts")


# Order Virus_extendedes by counts
order = df.sort_values("counts", ascending=False)["Virus_extended"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(4, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus_extended", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Clonotypes in both dataset")
plt.xlabel("")
# plt.title("Clones present in both the main\n and multimer-sort datasets")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/25_NumberOfMatchedClones.pdf")

## How many clones were present in CSF?

In [ ]:
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df = obs_df[obs_df["Virus_extended"] != "Cancer"]

df = obs_df[["clone_handle", "Virus_extended"]].drop_duplicates().value_counts().reset_index(name = "counts")
df = df[["Virus_extended"]].value_counts().reset_index(name="counts")

# Order Virus_extendedes by counts
order = df.sort_values("counts", ascending=False)["Virus_extended"]

# Initialize the matplotlib figure
f, ax = plt.subplots(figsize=(3.2, 4))

# Plot
sb.set_color_codes("pastel")
sb.barplot(x="Virus_extended", y="counts", data=df, order=order)

# Add count labels on top of bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=11
    )


# Add a legend and informative axis label
ax.legend(ncol=1, loc="lower right", frameon=True)
ax.set(ylabel="Clonotypes in both dataset")
plt.xlabel("")
# plt.title("Clones present in both the main\n and multimer-sort datasets")
plt.xticks(rotation=90)
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/25_NumberOfMatchedClones_InCSF.pdf")

# Phenotype of B-LCL reactive clones

In [ ]:
# Remove stored colors: 
mdata["gex"].uns.pop("LCL_reactive_colors", None)

In [ ]:
#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## annotate
mdata["gex"].obs["clone_lcl_handle"] = mdata["gex"].obs["donor"].astype(str) + "_" + mdata["gex"].obs["IR_VDJ_1_junction_aa"].astype(str) 
mdata["gex"].obs["LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (mdata["gex"].obs["catpat"]=="MS"), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

In [ ]:
#### annotate LCL reactive
LCL_list = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
LCL_list = LCL_list["Clone_handle"].drop_duplicates().tolist()

## annotate
mdata["gex"].obs["clone_lcl_handle"] = mdata["gex"].obs["donor"].astype(str) + "_" + mdata["gex"].obs["IR_VDJ_1_junction_aa"].astype(str) 
mdata["gex"].obs["LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (mdata["gex"].obs["catpat"]=="NIC"), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

## LCL-reactive yet +/- annotated with a Virus-specificity

In [ ]:
mdata["gex"].obs["Unicorn_LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (mdata["gex"].obs["Virus_extended"].isna()), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["Unicorn_LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

In [ ]:
mdata["gex"].obs["Unicorn_InMS_LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (mdata["gex"].obs["Virus_extended"].isna())& (mdata["gex"].obs["catpat"]=="MS"), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["Unicorn_InMS_LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

In [ ]:
mdata["gex"].obs["Unicorn_InNIC_LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (mdata["gex"].obs["Virus_extended"].isna())& (mdata["gex"].obs["catpat"]=="NIC"), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["Unicorn_InNIC_LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

In [ ]:
mdata["gex"].obs["KnownAg_LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (~mdata["gex"].obs["Virus_extended"].isna()), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["KnownAg_LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

In [ ]:
mdata["gex"].obs["KnownAg_InMS_LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (~mdata["gex"].obs["Virus_extended"].isna()) & (mdata["gex"].obs["catpat"]=="MS"), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["KnownAg_InMS_LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

mdata["gex"].obs[mdata["gex"].obs["KnownAg_InMS_LCL_reactive"]=="LCL-reactive"][["clone_handle", "Virus_extended"]].value_counts()

In [ ]:
mdata["gex"].obs["KnownAg_InNIC_LCL_reactive"] = np.where((mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (~mdata["gex"].obs["Virus_extended"].isna()) & (mdata["gex"].obs["catpat"]=="NIC"), "LCL-reactive", "Non-reactive")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "LCL-reactive" : "#D3D3D3",
 "Non-reactive" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["KnownAg_InNIC_LCL_reactive"], 
    groups=["LCL-reactive"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

mdata["gex"].obs[mdata["gex"].obs["KnownAg_InNIC_LCL_reactive"]=="LCL-reactive"][["clone_handle", "Virus_extended"]].value_counts()

In [ ]:
mdata["gex"].obs["KnownAg_InNIC_LCL_nonReac"] = np.where((~mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (mdata["gex"].obs["Virus_extended"]=="EBV") & (mdata["gex"].obs["catpat"]=="NIC"), "NR_KnownAg", "Other")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "NR_KnownAg" : "#D3D3D3",
 "Other" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["KnownAg_InNIC_LCL_nonReac"], 
    groups=["NR_KnownAg"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

mdata["gex"].obs[mdata["gex"].obs["KnownAg_InNIC_LCL_nonReac"]=="NR_KnownAg"][["clone_handle", "Virus_extended"]].value_counts()

In [ ]:
mdata["gex"].obs["KnownAg_InMS_LCL_nonReac"] = np.where((~mdata["gex"].obs["clone_lcl_handle"].isin(LCL_list)) & (mdata["gex"].obs["Virus_extended"]=="EBV") & (mdata["gex"].obs["catpat"]=="MS"), "NR_KnownAg", "Other")

## Visualize
# --- Display virus-specificities --- 
sc.settings.set_figure_params(figsize=(4, 4))

LCL_reactive = {
 "NR_KnownAg" : "#D3D3D3",
 "Other" : "#8B0000",
}

p1 = sc.pl.umap(
    mdata["gex"], 
    color=["KnownAg_InMS_LCL_nonReac"], 
    groups=["NR_KnownAg"],
    size=40, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    # palette=LCL_reactive,
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
# p1.savefig("../../05_plots/13_final_plots/52_VirusExtend_MS+NIC_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

mdata["gex"].obs[mdata["gex"].obs["KnownAg_InMS_LCL_nonReac"]=="NR_KnownAg"][["clone_handle", "Virus_extended"]].value_counts()

# Differential gene expression analysis 
## EBV-MS vs -NIC

## PB

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
mdata_csf = mdata.copy()
mu.pp.filter_obs(mdata_csf["gex"], "Compartment", lambda x: (x == "PB")) 
mu.pp.filter_obs(mdata_csf["gex"], "Virus_extended", lambda x: (x == "EBV")) 
mu.pp.filter_obs(mdata_csf["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"]))) 

#Drop unusedcategories
for col in mdata_csf["gex"].obs.select_dtypes(include="category").columns:
    mdata_csf["gex"].obs[col] = mdata_csf["gex"].obs[col].cat.remove_unused_categories()

## Define the filtered to alter: 
mdata_csf_filtered = mdata_csf["gex"].copy()

# --- Filter genes before differential gene expression ---
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cells, B cells and interacting cells

## --- Expression in NIC
adata = mdata_csf["gex"][mdata_csf["gex"].obs["catpat"] == "NIC"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## --- Expression in PB-biased T cells
adata = mdata_csf["gex"][mdata_csf["gex"].obs["catpat"] == "MS"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_set_expressed = set(gene_list_10pct_1+gene_list_10pct_2)
print(len(gene_set_expressed))

# -----------------------------
# Step 2: exclude unwanted genes
# -----------------------------
genes = mdata_csf["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-", "AC004448.2"))
exclude_genes = set(genes[exclude_mask])
genes_to_keep = sorted(gene_set_expressed - exclude_genes)
print("Final genes:", len(genes_to_keep))

# -----------------------------
# Step 3: subset AnnData correctly
# -----------------------------
mdata_csf_filtered = mdata_csf_filtered[:, genes_to_keep].copy()

# Perform naive differential gene expression
sc.tl.rank_genes_groups(mdata_csf_filtered, groupby="catpat", groups=["NIC"], reference="MS", key_added="upNIC_EBV", method="wilcoxon")
sc.tl.rank_genes_groups(mdata_csf_filtered, groupby="catpat", groups=["MS"], reference="NIC", key_added="upMS_EBV", method="wilcoxon")

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
     mdata_csf_filtered, groupby="catpat", standard_scale="var", n_genes=30, key="upNIC_EBV")
sc.pl.rank_genes_groups_dotplot(
     mdata_csf_filtered, groupby="catpat", standard_scale="var", n_genes=30, key="upMS_EBV")


In [ ]:
# %%% 
# Vulcano plot function
# %%%

# # From: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
LOG_FOLD_CHANGE = 0.5
PVALUE = -np.log10(0.05)  #Higher than p_adj 0.05. 
result = sc.get.rank_genes_groups_df(mdata_csf_filtered, group=None, key="upMS_EBV")
Add_labels = result[(result["pvals_adj"] < 0.05)]["names"].tolist()


# Add_labels = ['GZMK', 'CCR7', "GAPDH", "CCR5", 'CD27', "CD28", 'GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", "AC004448.2", "CLDND1", "CCL5", "KIR2DL4", "KIR2DL1", "PASK", "DKK3", "COTL1", "SARAF", "PPP1R14B", "NKG7", 
#               'GZMB', 'ZNF683', 'GZMH', 'KLRD1',  'CX3CR1', 'FNDC3B',  'KLRC2', "TYROBP", "AOAH", "GBP5", "S100A4", "EFHD1", "HLA-C", "EFHD2", "MYL12A", "ST8SIA1", "ITGB1", "LGALS1", "GNLY", "CD52", "IL32", "IRF1", "HOPX", 
#               "ZEB2", "FCRL6", "PRF1", "NKG7", 
#              ]

VOLCANO_PALETTE = {
    "neg": "#D89A52",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}

def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()
    print(result)

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 0.25) | (result["logfoldchanges"] < -0.25)]

    #Drop stuff that is na
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.grid(True, zorder=0)

    # Non-significant genes
    sb.regplot(
        x=nonsig["logfoldchanges"],
        y=nonsig["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color=palette["ns"]
    )

    # Positively enriched
    sb.regplot(
        x=sig_pos["logfoldchanges"],
        y=sig_pos["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["pos"],
        label="Positive enrichment"
    )

    # Negatively enriched
    sb.regplot(
        x=sig_neg["logfoldchanges"],
        y=sig_neg["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["neg"],
        label="Negative enrichment"
    )

    # Rasterize only the dots
    # for coll in ax.collections:
    #     coll.set_rasterized(True)

    # Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log padj")

    #Set limits
    ax.set_xlim(-6, 6)
    ax.set_ylim(-0.25, 5)

    if title is None:
        title = group_key.replace("_", " ")
    ax.set_title(title)

    ax.legend(frameon=False)
    sb.despine()
    ax.grid(False)
    plt.tight_layout()

    plt.savefig(
        "../../05_plots/13_final_plots/17_CD8_vulcanoPlot_CSFvPB_bias.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=400
    )
    plt.show()

volcano_plot(mdata_csf_filtered, group_key="upMS_EBV", title="<-- up NIC & up MS -->") 



# Report DEGs in PB

In [ ]:
result = sc.get.rank_genes_groups_df(mdata_csf_filtered, group=None, key="upMS_EBV")

#Print table
result[(result["pvals_adj"] < 0.05)].head(10)

In [ ]:
marker_genes = result[(result["pvals_adj"] < 0.05)]["names"].tolist()

donor_order = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]

mdata_csf["gex"].obs["donor"] = pd.Categorical(mdata_csf["gex"].obs["donor"], ordered=True, categories=donor_order)

dp = sc.pl.dotplot(
    mdata_csf["gex"],
    marker_genes,
    groupby="donor",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=True,
    show=False,
    return_fig=True,
)

dp.show()

In [ ]:
#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]


## CSF

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
mdata_csf = mdata.copy()
mu.pp.filter_obs(mdata_csf["gex"], "Compartment", lambda x: (x == "CSF")) 
mu.pp.filter_obs(mdata_csf["gex"], "Virus_extended", lambda x: (x == "EBV")) 
mu.pp.filter_obs(mdata_csf["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"]))) 

#Drop unusedcategories
for col in mdata_csf["gex"].obs.select_dtypes(include="category").columns:
    mdata_csf["gex"].obs[col] = mdata_csf["gex"].obs[col].cat.remove_unused_categories()

## Define the filtered to alter: 
mdata_csf_filtered = mdata_csf["gex"].copy()

# --- Filter genes before differential gene expression ---
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cells, B cells and interacting cells

## --- Expression in NIC
adata = mdata_csf["gex"][mdata_csf["gex"].obs["catpat"] == "NIC"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## --- Expression in PB-biased T cells
adata = mdata_csf["gex"][mdata_csf["gex"].obs["catpat"] == "MS"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_set_expressed = set(gene_list_10pct_1+gene_list_10pct_2)
print(len(gene_set_expressed))

# -----------------------------
# Step 2: exclude unwanted genes
# -----------------------------
genes = mdata_csf["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-", "AC004448.2"))
exclude_genes = set(genes[exclude_mask])
genes_to_keep = sorted(gene_set_expressed - exclude_genes)
print("Final genes:", len(genes_to_keep))

# -----------------------------
# Step 3: subset AnnData correctly
# -----------------------------
mdata_csf_filtered = mdata_csf_filtered[:, genes_to_keep].copy()

# Perform naive differential gene expression
sc.tl.rank_genes_groups(mdata_csf_filtered, groupby="catpat", groups=["NIC"], reference="MS", key_added="upNIC_EBV", method="wilcoxon")
sc.tl.rank_genes_groups(mdata_csf_filtered, groupby="catpat", groups=["MS"], reference="NIC", key_added="upMS_EBV", method="wilcoxon")

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
     mdata_csf_filtered, groupby="catpat", standard_scale="var", n_genes=30, key="upNIC_EBV")
sc.pl.rank_genes_groups_dotplot(
     mdata_csf_filtered, groupby="catpat", standard_scale="var", n_genes=30, key="upMS_EBV")


In [ ]:
# %%% 
# Vulcano plot function
# %%%

# # From: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
LOG_FOLD_CHANGE = 0.5
PVALUE = -np.log10(0.05)  #Higher than p_adj 0.05. 
result = sc.get.rank_genes_groups_df(mdata_csf_filtered, group=None, key="upMS_EBV")
Add_labels = result[(result["pvals_adj"] < 0.05)]["names"].tolist()


# Add_labels = ['GZMK', 'CCR7', "GAPDH", "CCR5", 'CD27', "CD28", 'GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", "AC004448.2", "CLDND1", "CCL5", "KIR2DL4", "KIR2DL1", "PASK", "DKK3", "COTL1", "SARAF", "PPP1R14B", "NKG7", 
#               'GZMB', 'ZNF683', 'GZMH', 'KLRD1',  'CX3CR1', 'FNDC3B',  'KLRC2', "TYROBP", "AOAH", "GBP5", "S100A4", "EFHD1", "HLA-C", "EFHD2", "MYL12A", "ST8SIA1", "ITGB1", "LGALS1", "GNLY", "CD52", "IL32", "IRF1", "HOPX", 
#               "ZEB2", "FCRL6", "PRF1", "NKG7", 
#              ]

VOLCANO_PALETTE = {
    "neg": "#D89A52",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}

def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()
    print(result)

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 0.25) | (result["logfoldchanges"] < -0.25)]

    #Drop stuff that is na
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.grid(True, zorder=0)

    # Non-significant genes
    sb.regplot(
        x=nonsig["logfoldchanges"],
        y=nonsig["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color=palette["ns"]
    )

    # Positively enriched
    sb.regplot(
        x=sig_pos["logfoldchanges"],
        y=sig_pos["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["pos"],
        label="Positive enrichment"
    )

    # Negatively enriched
    sb.regplot(
        x=sig_neg["logfoldchanges"],
        y=sig_neg["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["neg"],
        label="Negative enrichment"
    )

    # Rasterize only the dots
    # for coll in ax.collections:
    #     coll.set_rasterized(True)

    # Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log padj")

    #Set limits
    ax.set_xlim(-6, 6)
    ax.set_ylim(-0.25, 5)

    if title is None:
        title = group_key.replace("_", " ")
    ax.set_title(title)

    ax.legend(frameon=False)
    sb.despine()
    ax.grid(False)
    plt.tight_layout()

    plt.savefig(
        "../../05_plots/13_final_plots/17_CD8_vulcanoPlot_CSFvPB_bias.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=400
    )
    plt.show()

volcano_plot(mdata_csf_filtered, group_key="upMS_EBV", title="<-- up NIC & up MS -->") 



# Report DEGs in CSF

In [ ]:
result = sc.get.rank_genes_groups_df(mdata_csf_filtered, group=None, key="upMS_EBV")
result[(result["pvals_adj"] < 0.05) & (result["logfoldchanges"] > 0)]


In [ ]:
marker_genes = result[(result["pvals_adj"] < 0.05)]["names"].tolist()

donor_order = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]

mdata_csf["gex"].obs["donor"] = pd.Categorical(mdata_csf["gex"].obs["donor"], ordered=True, categories=donor_order)

dp = sc.pl.dotplot(
    mdata_csf["gex"],
    marker_genes,
    groupby="donor",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=True,
    show=False,
    return_fig=True,
)

dp.show()

# Pseudobulk - MS vs NIC 

### Pseudobulked using Decoupler's version of Deseq2
https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html

The pseudo-bulk approach involves the following steps:

    Subsetting the cell type of interest

    Extracting their raw integer counts

    Summing their counts per gene into a single profile if they pass quality NIC

Then, DEA can be performed if there are at least two biological replicates per condition (more replicates are recommended).

Pseudobulking can easily be performed using the function decoupler.pp.pseudobulk(). In this example, the counts are just summed, though other modes such as the mean or any custom aggregation function are available. For more information, refer to the mode argument.

In [ ]:
mdata["gex"].obs[["catpat", "Virus_extended"]].value_counts()

In [ ]:
## ---- Define the data ---- 
sdata_bulk = mdata.copy()

mu.pp.filter_obs(sdata_bulk["gex"], "donor", lambda x: ~x.isin(["Pt4", "Pt5"]))
mu.pp.filter_obs(sdata_bulk["gex"], "Compartment", lambda x: x == "PB")
mu.pp.filter_obs(sdata_bulk["gex"], "Virus_extended", lambda x: (x == "EBV")) 

#Drop unusedcategories
for col in sdata_bulk["gex"].obs.select_dtypes(include="category").columns:
    sdata_bulk["gex"].obs[col] = sdata_bulk["gex"].obs[col].cat.remove_unused_categories()

# Create a mask for genes to exclude
genes = sdata_bulk["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_bulk = sdata_bulk["gex"][:, genes_to_keep].copy()

## Bring the raw counts into .X
sdata_bulk.X = sdata_bulk.layers["counts"].copy()

In [ ]:
## ---- Use decoupler to pseudobulk cells ----
pdata = dc.pp.pseudobulk(
    adata=sdata_bulk,
    sample_col="donor",
    groups_col="catpat",
    mode="sum",
)

pdata

In [ ]:
pdata.obs

In [ ]:
dc.pp.filter_samples(pdata, min_cells=2, min_counts=1000)

In [ ]:
pdata.obs

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=pdata,
    design_factors=["catpat"],
    refit_cooks=True,
    inference=inference,
)

# Compute LFCs
dds.deseq2()

# Extract contrast between conditions
stat_res = DeseqStats(dds, contrast=["catpat", "MS", "NIC"], inference=inference)

# Compute Wald test
stat_res.summary()

In [ ]:
# Extract results
results_df = stat_res.results_df

# filter
results_df = results_df[~results_df["pvalue"].isna()]
results_df = results_df[results_df["baseMean"] > 10]
results_df = results_df[results_df.index != "AC004448.2"] #Exclude a donor-specific transcript, that is strongly expressed in select donors
# results_df[results_df["padj"] < 0.05].sort_values("log2FoldChange", ascending=False).head(50)


In [ ]:
results_df.sort_values("pvalue", ascending=True).head(10)

In [ ]:
dc.pl.volcano(results_df, x="log2FoldChange", y="pvalue", top=20, figsize =(6,6))
plt.savefig("../../05_plots/13_final_plots/49_MSvNIC_EBV-specific_PB.pdf", dpi=400, format="pdf")
plt.show()

In [ ]:
marker_genes = results_df[results_df["pvalue"] < 0.05].reset_index()["genes"].tolist()

donor_order = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
mdata_pb = mdata.copy()
mu.pp.filter_obs(mdata_pb["gex"], "Compartment", lambda x: (x == "CSF")) 
mu.pp.filter_obs(mdata_pb["gex"], "Virus_extended", lambda x: (x == "EBV")) 
mu.pp.filter_obs(mdata_pb["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"]))) 

#Drop unusedcategories
for col in mdata_pb["gex"].obs.select_dtypes(include="category").columns:
    mdata_pb["gex"].obs[col] = mdata_pb["gex"].obs[col].cat.remove_unused_categories()
mdata_pb["gex"].obs["donor"] = pd.Categorical(mdata_pb["gex"].obs["donor"], ordered=True, categories=donor_order)

dp = sc.pl.dotplot(
    mdata_pb["gex"],
    marker_genes,
    groupby="donor",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=True,
    show=False,
    return_fig=True,
)

dp.show()

# Graveyard

In [ ]:
## Get the obs frame
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
# obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

#Annotated non-specific cells or clones with unknown specificity / non-annotated
obs_df["Virus_extended"] = obs_df["Virus_extended"].astype(str)
obs_df["Virus_extended"] = obs_df["Virus_extended"].fillna("Not annotated")

## Annotation of shared clones
CSF_clones = obs_df[obs_df["Compartment"]=="CSF"]["clone_handle"].drop_duplicates().tolist()
PB_clones = obs_df[obs_df["Compartment"]=="PB"]["clone_handle"].drop_duplicates().tolist()
shared_clones= set(CSF_clones) & set(PB_clones)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["donor", "catpat", "Virus_extended", "Compartment"], observed=True)
    .size()
    .reset_index(name="count")
)
print(count_df.sort_values("donor"))
# Total per cat_shared
count_df["total_cells"] = count_df.groupby(["Compartment", "donor", "catpat"], observed=True)["count"].transform("sum")

# Percentage
count_df["virus_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index=["Compartment", "donor", "catpat"], columns="Virus_extended", values="virus_pct")
    .fillna(0)
    .reset_index()
    .sort_values("Compartment")
)


plot_df= plot_df.sort_values("Compartment")

Virus_interest = "EBV"
plot_df2 = plot_df[["Compartment", "donor", "catpat", Virus_interest]]

#Drop unusedcategories
for col in plot_df2.select_dtypes(include="category").columns:
    plot_df2[col] = plot_df2[col].cat.remove_unused_categories()

# Plot per donor
fig, ax = plt.subplots(figsize=(4, 4))

# Boxplot (foreground)
sb.boxplot(
    data=plot_df2,
    x="Compartment",
    y=Virus_interest,
    hue="catpat",
    width=0.75,
    color="white",
    order=["CSF", "PB"],
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=plot_df2,
    x="Compartment",
    y=Virus_interest,
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    order=["CSF", "PB"],
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

ax.set_ylabel(f"% {Virus_interest}-specific cells")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
ax.set_title("% of cells. n > 0 cells")

# Remove duplicate legend from stripplot
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    title=""
)

sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
## Get the obs frame
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
# obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

## Annotation of shared clones
CSF_clones = obs_df[obs_df["Compartment"]=="CSF"]["clone_handle"].drop_duplicates().tolist()
PB_clones = obs_df[obs_df["Compartment"]=="PB"]["clone_handle"].drop_duplicates().tolist()
shared_clones= set(CSF_clones) & set(PB_clones)

# -----------------------------
# Create grouping variables
# -----------------------------
obs_df["x_cat"] = obs_df["Compartment"].astype(str) + "_" + obs_df["catpat"].astype(str)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df[["clone_handle", "x_cat", "Virus_extended"]].
    drop_duplicates()
    .groupby(["x_cat", "Virus_extended"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per cat_shared
count_df["total_cells"] = count_df.groupby("x_cat")["count"].transform("sum")

# Percentage
count_df["virus_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index="x_cat", columns="Virus_extended", values="virus_pct")
    .fillna(0)
    .reset_index()
    .sort_values("x_cat")
)


plot_df= plot_df.sort_values("x_cat")
plot_df

In [ ]:
# -----------------------------
# Plot virus distribution per clone-sharing category
# -----------------------------
# Optional order
x_order = ["CSF_NIC", "CSF_MS","PB_NIC", "PB_MS"]
plot_df["x_cat"] = pd.Categorical(
    plot_df["x_cat"],
    categories=x_order,
    ordered=True
)
plot_df = plot_df.sort_values("x_cat").reset_index(drop=True)

#plot
x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(6, 4))
bottom = np.zeros(len(plot_df))

for virus in virus_palette.keys():
    if virus in plot_df.columns:
        ax.bar(
            x,
            plot_df[virus],
            bottom=bottom,
            color=virus_palette[virus],
            label=virus
        )
        bottom += plot_df[virus].values

ax.set_xticks(x)
ax.set_xticklabels(plot_df["x_cat"], rotation=45, ha="right")

ax.set_ylabel("% of clones")
ax.set_xlabel("")
ax.set_ylim(0, 100)

plt.title("Clonal composition (n > 0 cells)")

ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
## Get the obs frame
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
# obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

#Annotated non-specific cells or clones with unknown specificity / non-annotated
obs_df["Virus_extended"] = obs_df["Virus_extended"].astype(str)
obs_df["Virus_extended"] = obs_df["Virus_extended"].fillna("Not annotated")

## Annotation of shared clones
CSF_clones = obs_df[obs_df["Compartment"]=="CSF"]["clone_handle"].drop_duplicates().tolist()
PB_clones = obs_df[obs_df["Compartment"]=="PB"]["clone_handle"].drop_duplicates().tolist()
shared_clones= set(CSF_clones) & set(PB_clones)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df[["clone_handle", "donor", "catpat", "Compartment", "Virus_extended"]].
    drop_duplicates()
    .groupby(["donor", "catpat", "Virus_extended", "Compartment"], observed=True)
    .size()
    .reset_index(name="count")
)
print(count_df.sort_values("donor"))
# Total per cat_shared
count_df["total_cells"] = count_df.groupby(["Compartment", "donor", "catpat"], observed=True)["count"].transform("sum")

# Percentage
count_df["virus_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index=["Compartment", "donor", "catpat"], columns="Virus_extended", values="virus_pct")
    .fillna(0)
    .reset_index()
    .sort_values("Compartment")
)


plot_df= plot_df.sort_values("Compartment")

Virus_interest = "EBV"
plot_df2 = plot_df[["Compartment", "donor", "catpat", Virus_interest]]

#Drop unusedcategories
for col in plot_df2.select_dtypes(include="category").columns:
    plot_df2[col] = plot_df2[col].cat.remove_unused_categories()

# Plot per donor
fig, ax = plt.subplots(figsize=(4, 4))

# Boxplot (foreground)
sb.boxplot(
    data=plot_df2,
    x="Compartment",
    y=Virus_interest,
    hue="catpat",
    width=0.75,
    color="white",
    order=["CSF", "PB"],
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=plot_df2,
    x="Compartment",
    y=Virus_interest,
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    order=["CSF", "PB"],
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

ax.set_ylabel(f"% {Virus_interest}-specific clones")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
ax.set_title("% of clones. n > 0 clones")

# Remove duplicate legend from stripplot
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    title=""
)

sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

# Percentage of cells in CSF

In [ ]:
# -----------------------------
# Filter
# -----------------------------
obs_df = mdata["gex"].obs.dropna(subset=["Virus_extended"])
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 12]
obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

# -----------------------------
# Create grouping variable
# -----------------------------
obs_df["x_cat"] = obs_df["Virus_extended"].astype(str) + "_" + obs_df["catpat"].astype(str)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["x_cat", "Compartment"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per x_cat
count_df["total_cells"] = count_df.groupby("x_cat")["count"].transform("sum")

# Percentage
count_df["comp_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index="x_cat", columns="Compartment", values="comp_pct")
    .fillna(0)
    .reset_index()
    .sort_values("x_cat")
)

x_order = ["CMV_NIC", "CMV_MS", "EBV_NIC", "EBV_MS", "FLU-A_NIC", "FLU-A_MS", "SARS-COV-2_NIC", "SARS-COV-2_MS"]

# Reorder according to desired x-axis order
plot_df["x_cat"] = pd.Categorical(
    plot_df["x_cat"],
    categories=x_order,
    ordered=True
)

plot_df = plot_df.sort_values("x_cat")

# -----------------------------
# Plot
# -----------------------------
csf_color = "#4B8AC1"   # muted blue
pb_color  = "#C06A6A"   # muted orange
x = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(6, 4))

# CSF (bottom)
ax.bar(
    x,
    plot_df["CSF"],
    color=csf_color,
    label="CSF"
)

# PB (correct stacking!)
ax.bar(
    x,
    plot_df["PB"],
    bottom=plot_df["CSF"],
    color=pb_color,
    label="PB"
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["x_cat"], rotation=45, ha="right")

ax.set_ylabel("% of cells")
ax.set_xlabel("Condition")
plt.title("% cells in each compartment")

ax.legend(frameon=False)
sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
### Per clone
# -----------------------------
# Filter
# -----------------------------
obs_df = mdata["gex"].obs.dropna(subset=["Virus_extended"])
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

## Percentage of cells in CSF per clone
count_df = obs_df[["clone_handle", "Virus_extended", "donor", "catpat", "Compartment", "type_x_clone_count", "total_clone_count"]].drop_duplicates()

## collapse to one value per donor
count_df = (count_df
    .groupby(["Virus_extended", "donor", "catpat", "Compartment"],observed=True)
    .agg(cells_comp=("type_x_clone_count", "sum"), total_cells=("total_clone_count", "sum"))
    .reset_index()
)

# percentage cells per donor
count_df["comp_pct"] = 100*count_df["cells_comp"] / count_df["total_cells"]

#Drop unusedcategories
for col in count_df.select_dtypes(include="category").columns:
    count_df[col] = count_df[col].cat.remove_unused_categories()

#Focus on CSF
count_df = count_df[count_df["Compartment"]=="CSF"]

## Plot per donor
fig, ax = plt.subplots(figsize=(8, 4))

# Boxplot (foreground)
sb.boxplot(
    data=count_df,
    x="Virus_extended",
    y="comp_pct",
    hue="catpat",
    width=0.75,
    color="white",
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=count_df,
    x="Virus_extended",
    y="comp_pct",
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

ax.set_ylabel("% cells in CSF per donor")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)

# Remove duplicate legend from stripplot
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    title=""
)

sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
count_df[count_df["Virus_extended"]=="CMV"]

# Do EBV-specific T cells make up a greater proportion of CSF in MS?

In [ ]:
## Get the obs frame
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
# obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

## Annotation of shared clones
CSF_clones = obs_df[obs_df["Compartment"]=="CSF"]["clone_handle"].drop_duplicates().tolist()
PB_clones = obs_df[obs_df["Compartment"]=="PB"]["clone_handle"].drop_duplicates().tolist()
shared_clones= set(CSF_clones) & set(PB_clones)

# -----------------------------
# Create grouping variables
# -----------------------------
obs_df["cat_shared"] = np.where(obs_df["clone_handle"].isin(shared_clones), "Shared clones", 
                               np.where(obs_df["clone_handle"].isin(CSF_clones), "CSF only", 
                                       np.where(obs_df["clone_handle"].isin(PB_clones), "PB-only", "error")))

obs_df["x_cat"] = obs_df["cat_shared"].astype(str) + "_" + obs_df["catpat"].astype(str)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["x_cat", "Virus_extended"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per cat_shared
count_df["total_cells"] = count_df.groupby("x_cat")["count"].transform("sum")

# Percentage
count_df["virus_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index="x_cat", columns="Virus_extended", values="virus_pct")
    .fillna(0)
    .reset_index()
    .sort_values("x_cat")
)


plot_df= plot_df.sort_values("x_cat")
plot_df

In [ ]:
# -----------------------------
# Plot virus distribution per clone-sharing category
# -----------------------------

# Define virus colors
virus_palette = {
    "5-OP-RU": "#7B6FD0",      # purple
    "CMV": "#4C78A8",          # blue
    "Cancer": "#E45756",       # red
    "EBV": "#72B7B2",           # teal
    "FLU-A": "#F2CF5B",         # yellow
    "HAdV-C": "#54A24B",        # green
    "HHV-1": "#B279A2",         # mauve
    "HIV": "#FF9DA6",           # pink
    "HPV": "#9D755D",           # brown
    "SARS-COV-2": "#FF9F40",    # orange
    "VZV": "#000000",           # light teal
    "YFV": "#BAB0AC"            # grey
}

# Optional order
x_order = ["CSF only_NIC", "CSF only_MS", "Shared clones_NIC", "Shared clones_MS", "PB-only_NIC", "PB-only_MS"]
plot_df["x_cat"] = pd.Categorical(
    plot_df["x_cat"],
    categories=x_order,
    ordered=True
)
plot_df = plot_df.sort_values("x_cat").reset_index(drop=True)

#plot
x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(6, 4))
bottom = np.zeros(len(plot_df))

for virus in virus_palette.keys():
    if virus in plot_df.columns:
        ax.bar(
            x,
            plot_df[virus],
            bottom=bottom,
            color=virus_palette[virus],
            label=virus
        )
        bottom += plot_df[virus].values

ax.set_xticks(x)
ax.set_xticklabels(plot_df["x_cat"], rotation=45, ha="right")

ax.set_ylabel("% of cells")
ax.set_xlabel("")
ax.set_ylim(0, 100)

plt.title("Composition (n > 0 cells)")

ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")

plt.tight_layout()
plt.show()

# Clonal composition for EBV-specifiic cells - per donor

In [ ]:
## Get the obs frame
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
# obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

#Annotated non-specific cells or clones with unknown specificity / non-annotated
# obs_df["Virus_extended"] = obs_df["Virus_extended"].astype(str)
# obs_df["Virus_extended"] = obs_df["Virus_extended"].fillna("Not annotated")

## Annotation of shared clones
CSF_clones = obs_df[obs_df["Compartment"]=="CSF"]["clone_handle"].drop_duplicates().tolist()
PB_clones = obs_df[obs_df["Compartment"]=="PB"]["clone_handle"].drop_duplicates().tolist()
shared_clones= set(CSF_clones) & set(PB_clones)

# -----------------------------
# Create grouping variables
# -----------------------------
obs_df["cat_shared"] = np.where(obs_df["clone_handle"].isin(shared_clones), "Shared clones", 
                               np.where(obs_df["clone_handle"].isin(CSF_clones), "CSF only", 
                                       np.where(obs_df["clone_handle"].isin(PB_clones), "PB-only", "error")))


# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df
    .groupby(["cat_shared", "donor", "catpat", "Virus_extended"], observed=True)
    .size()
    .reset_index(name="count")
)
print(count_df.sort_values("donor"))
# Total per cat_shared
count_df["total_cells"] = count_df.groupby(["cat_shared", "donor", "catpat"], observed=True)["count"].transform("sum")

# Percentage
count_df["virus_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index=["cat_shared", "donor", "catpat"], columns="Virus_extended", values="virus_pct")
    .fillna(0)
    .reset_index()
    .sort_values("cat_shared")
)


plot_df= plot_df.sort_values("cat_shared")

Virus_interest = "EBV"
plot_df2 = plot_df[["cat_shared", "donor", "catpat", Virus_interest]]

#Drop unusedcategories
for col in plot_df2.select_dtypes(include="category").columns:
    plot_df2[col] = plot_df2[col].cat.remove_unused_categories()

# Plot per donor
fig, ax = plt.subplots(figsize=(8, 4))

# Boxplot (foreground)
sb.boxplot(
    data=plot_df2,
    x="cat_shared",
    y=Virus_interest,
    hue="catpat",
    width=0.75,
    color="white",
    order=["CSF only", "Shared clones", "PB-only"],
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=plot_df2,
    x="cat_shared",
    y=Virus_interest,
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    order=["CSF only", "Shared clones", "PB-only"],
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

ax.set_ylabel(f"% {Virus_interest}-specific cells")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)

# Remove duplicate legend from stripplot
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    title=""
)

sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
obs_df[obs_df["Virus_extended"]!="nan"][["donor", "catpat"]].value_counts().reset_index().sort_values("donor")

# Do EBV-specific CD8 T cell clonal composition change?

In [ ]:
## Get the obs frame
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
# obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

## Annotation of shared clones
CSF_clones = obs_df[obs_df["Compartment"]=="CSF"]["clone_handle"].drop_duplicates().tolist()
PB_clones = obs_df[obs_df["Compartment"]=="PB"]["clone_handle"].drop_duplicates().tolist()
shared_clones= set(CSF_clones) & set(PB_clones)

# -----------------------------
# Create grouping variables
# -----------------------------
obs_df["cat_shared"] = np.where(obs_df["clone_handle"].isin(shared_clones), "Shared clones", 
                               np.where(obs_df["clone_handle"].isin(CSF_clones), "CSF only", 
                                       np.where(obs_df["clone_handle"].isin(PB_clones), "PB-only", "error")))

obs_df["x_cat"] = obs_df["cat_shared"].astype(str) + "_" + obs_df["catpat"].astype(str)

# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
    obs_df[["clone_handle", "x_cat", "Virus_extended"]].
    drop_duplicates()
    .groupby(["x_cat", "Virus_extended"], observed=True)
    .size()
    .reset_index(name="count")
)

# Total per cat_shared
count_df["total_clones"] = count_df.groupby("x_cat")["count"].transform("sum")

# Percentage
count_df["virus_pct"] = 100 * count_df["count"] / count_df["total_clones"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index="x_cat", columns="Virus_extended", values="virus_pct")
    .fillna(0)
    .reset_index()
    .sort_values("x_cat")
)


plot_df= plot_df.sort_values("x_cat")
plot_df

In [ ]:
# -----------------------------
# Plot virus distribution per clone-sharing category
# -----------------------------

# Define virus colors
virus_palette = {
    "5-OP-RU": "#7B6FD0",      # purple
    "CMV": "#4C78A8",          # blue
    "Cancer": "#E45756",       # red
    "EBV": "#72B7B2",           # teal
    "FLU-A": "#F2CF5B",         # yellow
    "HAdV-C": "#54A24B",        # green
    "HHV-1": "#B279A2",         # mauve
    "HIV": "#FF9DA6",           # pink
    "HPV": "#9D755D",           # brown
    "SARS-COV-2": "#FF9F40",    # orange
    "VZV": "#000000",           # light teal
    "YFV": "#BAB0AC"            # grey
}

# Optional order
x_order = ["CSF only_NIC", "CSF only_MS", "Shared clones_NIC", "Shared clones_MS", "PB-only_NIC", "PB-only_MS"]
plot_df["x_cat"] = pd.Categorical(
    plot_df["x_cat"],
    categories=x_order,
    ordered=True
)
plot_df = plot_df.sort_values("x_cat").reset_index(drop=True)

#plot
x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(6, 4))
bottom = np.zeros(len(plot_df))

for virus in virus_palette.keys():
    if virus in plot_df.columns:
        ax.bar(
            x,
            plot_df[virus],
            bottom=bottom,
            color=virus_palette[virus],
            label=virus
        )
        bottom += plot_df[virus].values

ax.set_xticks(x)
ax.set_xticklabels(plot_df["x_cat"], rotation=45, ha="right")

ax.set_ylabel("% of clones")
ax.set_xlabel("")
ax.set_ylim(0, 100)

plt.title("Clonal composition (n > 0 cells)")

ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")

plt.tight_layout()
plt.show()

# Clonal composition per donor

In [ ]:
## Get the obs frame
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["total_clone_count"] > 0]
# obs_df = obs_df[obs_df["Virus_extended"].isin(["CMV", "EBV", "FLU-A", "SARS-COV-2"])]

#Annotated non-specific cells or clones with unknown specificity / non-annotated
# obs_df["Virus_extended"] = obs_df["Virus_extended"].astype(str)
# obs_df["Virus_extended"] = obs_df["Virus_extended"].fillna("Not annotated")

## Annotation of shared clones
CSF_clones = obs_df[obs_df["Compartment"]=="CSF"]["clone_handle"].drop_duplicates().tolist()
PB_clones = obs_df[obs_df["Compartment"]=="PB"]["clone_handle"].drop_duplicates().tolist()
shared_clones= set(CSF_clones) & set(PB_clones)

# -----------------------------
# Create grouping variables
# -----------------------------
obs_df["cat_shared"] = np.where(obs_df["clone_handle"].isin(shared_clones), "Shared clones", 
                               np.where(obs_df["clone_handle"].isin(CSF_clones), "CSF only", 
                                       np.where(obs_df["clone_handle"].isin(PB_clones), "PB-only", "error")))

obs_df = obs_df[obs_df["Compartment"] == "PB"]


# -----------------------------
# Count cells per group
# -----------------------------
count_df = (
        obs_df[["clone_handle",  "donor", "catpat", "cat_shared", "Virus_extended"]].
    drop_duplicates()
    .groupby(["cat_shared", "donor", "catpat", "Virus_extended"], observed=True)
    .size()
    .reset_index(name="count")
)
print(count_df.sort_values("donor"))
# Total per cat_shared
count_df["total_cells"] = count_df.groupby(["cat_shared", "donor", "catpat"], observed=True)["count"].transform("sum")

# Percentage
count_df["virus_pct"] = 100 * count_df["count"] / count_df["total_cells"]

# -----------------------------
# Pivot
# -----------------------------
plot_df = (
    count_df
    .pivot(index=["cat_shared", "donor", "catpat"], columns="Virus_extended", values="virus_pct")
    .fillna(0)
    .reset_index()
    .sort_values("cat_shared")
)


plot_df= plot_df.sort_values("cat_shared")

Virus_interest = "EBV"
plot_df2 = plot_df[["cat_shared", "donor", "catpat", Virus_interest]]

#Drop unusedcategories
for col in plot_df2.select_dtypes(include="category").columns:
    plot_df2[col] = plot_df2[col].cat.remove_unused_categories()

# Plot per donor
fig, ax = plt.subplots(figsize=(8, 4))

# Boxplot (foreground)
sb.boxplot(
    data=plot_df2,
    x="cat_shared",
    y=Virus_interest,
    hue="catpat",
    width=0.75,
    color="white",
    order=["CSF only", "Shared clones", "PB-only"],
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

# Stripplot (background dots)
sb.stripplot(
    data=plot_df2,
    x="cat_shared",
    y=Virus_interest,
    hue="catpat",
    dodge=True,
    s=5,
    jitter=True,
    order=["CSF only", "Shared clones", "PB-only"],
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette={"MS" : "#000000", "NIC" : "#FFFFFF"},
    ax=ax
)

ax.set_ylabel(f"% {Virus_interest}-specific cells")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)

# Remove duplicate legend from stripplot
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    frameon=False,
    title=""
)

sb.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

# heatmap w. % freq per cluster